# Cassette fronts: R29 partial-front sampling

**Standalone delivery: upload this file to Colab. No unpublished GitHub commit is required.**

**Select an L4 or A100 GPU, choose Run all, and return `baomai-partial-front-sampling-r29-results.zip`.**

The PR105 review and diagnosis are complete. This notebook tests the next training hypothesis: **does extra exposure to existing partial-front images help, with the same training budget?**

Both roles use the accepted, eligible **R28 labels**, independently load the original epoch173 checkpoint, and train for 30 epochs with seed 46, batch 4, image size1280, FP32 and the existing AdamW recipe. The only intended intervention is image sampling. R29 names this experiment; it is not a new annotation revision.

The control sees each of the 137 training images once per epoch. The candidate sees nine declared partial-rich training images twice, replacing nine other eligible presentations on a rotating schedule. Both have 137 presentations per epoch, 1,050 batches and 4,110 presentations overall. The 13 empty scenes and 32 held/excluded-from-amendment views stay at one presentation per epoch. Every training image remains represented over the run. Actual exposures and optimizer updates are verified.

The chosen photos are `012, 018, 023, 031, 044, 045, 094, 116, 128`, each with the `new_000` prefix. Their selection uses training labels only; the 20 development photos never enter training. Repeating a photo repeats all its fronts and background. This is a bounded hypothesis, with no promised improvement.

Google Drive preserves verified completed models and the result ZIP. Complete the Drive authorization when requested. Keep both models together; completion does not automatically replace the retained PR35 model.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time

SOURCE_REVISION = "eea8e52a983f162512c4aa34ced73761b089609b"
PROTOCOL_PATH = "experiments/partial-front-sampling-2026-11-24-r29.json"
PROTOCOL_SHA256 = "2ab005b9d64bdd388a71a1026df4885152762be1f992cd90607d5256a046468f"
EXPERIMENT_ID = "partial-front-sampling-r29"
ROLE_RECIPES = {"control": "reviewed-incremental-fronts-seed46-v1",
                "candidate": "reviewed-incremental-fronts-seed46-v1"}
RECIPE_SHA256 = "f18e0a813a1057e1b3ab3a44ea3ce9b2a3e7673cc2bf04ca09b394bfdd8c51b8"
RUN_BASE = (Path("/content") if Path("/content").exists() else Path.cwd()) / "baomai-partial-front-sampling-r29"
ROLES = ("control", "candidate")

assert re.fullmatch(r"[0-9a-f]{40}", SOURCE_REVISION), "Use the final handoff notebook with its full source commit pinned."
assert re.fullmatch(r"[0-9a-f]{64}", PROTOCOL_SHA256), "Use the final handoff notebook with its protocol hash pinned."
assert sys.version_info >= (3, 11), "Python3.11 or newer is required."

def run(command, *, cwd=None, env=None):
    print("+", " ".join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

workspace_identity = {"source_revision": SOURCE_REVISION, "protocol_sha256": PROTOCOL_SHA256}
if RUN_BASE.exists() or RUN_BASE.is_symlink():
    assert RUN_BASE.is_dir() and not RUN_BASE.is_symlink(), "Choose a fresh nonsymlink workspace."
    marker = RUN_BASE / "workspace-identity.json"
    assert marker.is_file() and json.loads(marker.read_text()) == workspace_identity, "Existing workspace belongs to different pins; preserve it and choose a new RUN_BASE."
else:
    RUN_BASE.mkdir(parents=True)
    (RUN_BASE / "workspace-identity.json").write_text(json.dumps(workspace_identity, indent=2) + "\n")
REPO = RUN_BASE / "dl"
DRIVE_ROOT = None
try:
    from google.colab import drive
except ImportError:
    print("Drive is unavailable here; preserve completed bundles and the ZIP from the local workspace.")
else:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/baomai-partial-front-sampling-r29-20261124")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Drive folder:", DRIVE_ROOT)

def safe_member(name):
    p = Path(name)
    assert isinstance(name, str) and name and not p.is_absolute() and ".." not in p.parts
    assert "\\" not in name and ":" not in name and p.as_posix() == name
    return p

def artifact_inventory(path):
    path = Path(path)
    if path.is_file():
        names = [path.name]
        base = path.parent
    else:
        base = path
        receipt = json.loads((path / "training-receipt.json").read_text())
        names = ["training-receipt.json", "staging-manifest.json", receipt["log"]["path"]]
        if (path / "checkpoint-manifest.json").is_file():
            manifest = json.loads((path / "checkpoint-manifest.json").read_text())
            names += ["checkpoint-manifest.json", manifest["checkpoint"]["path"], receipt["results"]["path"]]
    assert len(names) == len(set(names))
    return [{"path": name, "sha256": sha256(base / safe_member(name)), "size_bytes": (base / name).stat().st_size}
            for name in sorted(names)]

def validate_copy(path, marker):
    base = path if marker["directory"] else path.parent
    for entry in marker["files"]:
        target = base / safe_member(entry["path"])
        assert target.is_file() and not target.is_symlink(), f"Missing backup artifact: {target}"
        assert target.stat().st_size == entry["size_bytes"] and sha256(target) == entry["sha256"], f"Backup hash mismatch: {target}"

def save_to_drive(path):
    if DRIVE_ROOT is None:
        return
    path = Path(path)
    relative = path.relative_to(RUN_BASE).as_posix()
    if relative == "resource-trial":
        relative = "resource-trials/" + sha256(path / "training-receipt.json")[:20]
    destination = DRIVE_ROOT / safe_member(relative)
    marker_path = Path(str(destination) + ".complete")
    marker = {**workspace_identity, "relative_path": relative, "directory": path.is_dir(), "files": artifact_inventory(path)}
    if marker_path.is_file():
        assert json.loads(marker_path.read_text()) == marker, f"Drive already has different completed evidence: {destination}"
        validate_copy(destination, marker)
        print("Verified existing Drive backup:", destination)
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Only an incomplete copy in this notebook's exact destination is replaced.
    if destination.is_dir():
        shutil.rmtree(destination)
    elif destination.exists():
        destination.unlink()
    if path.is_dir():
        destination.mkdir()
        for entry in marker["files"]:
            target = destination / safe_member(entry["path"])
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(path / entry["path"], target)
    else:
        shutil.copyfile(path, destination)
    validate_copy(destination, marker)
    marker_path.write_text(json.dumps(marker, indent=2) + "\n")
    print("Saved verified completed artifacts:", destination, flush=True)

def restore_from_drive(relative):
    if DRIVE_ROOT is None:
        return None
    source = DRIVE_ROOT / safe_member(relative)
    marker_path = Path(str(source) + ".complete")
    if not marker_path.is_file():
        return None
    marker = json.loads(marker_path.read_text())
    assert all(marker.get(k) == v for k, v in workspace_identity.items()), "Drive completion belongs to different source/protocol pins."
    assert marker["relative_path"] == relative
    validate_copy(source, marker)
    destination = RUN_BASE / safe_member(relative)
    if destination.exists() or destination.is_symlink():
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    if marker["directory"]:
        destination.mkdir()
        for entry in marker["files"]:
            target = destination / safe_member(entry["path"])
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source / entry["path"], target)
    else:
        assert len(marker["files"]) == 1 and marker["files"][0]["path"] == source.name
        shutil.copyfile(source, destination)
    validate_copy(destination, marker)
    print("Restored completed evidence:", destination, flush=True)
    return destination

print("Workspace:", RUN_BASE)

import base64
import zlib
BASE_REVISION = 'c487e7abbb0aca19f59fb143ca2320d3adb01b3e'
EMBEDDED_FILES = {
 "scripts/train_pose_comparison.py": {
  "sha256": "2ca801a601e3e20e228e43715e2e9d542648e84b2d1868d7846ff344640aa357",
  "size_bytes": 30197,
  "zlib_base64": "eNq9PWlz47iV3/0rGO6HUD0yfcyxXZpRqjrd7okrbttlu7M1pVWxKAmyGVMkw8PHOP7v+w4ABEmQkjvJdiVjiQQeHh4e3g3ov/5wUBX5wSJKDkTy4GTP5V2afL/nuu5VlTihs0irZCVWTi6KtMqXwinzKIydNHfSBL+ESxEuYuEswnQTRs46SsR+WSVRcuuE+cYHOHt70SZL89Ipnos9+L+/SpMyeMyjUgSL51Is05Vwps5NXgndNMxvszAvxN46TzfOKixFGW2EI9+q72MH//s7IKL68Z84WvgbUYbQLlRv/l6kifqcFgw3C8s7aKvAXsJX1SSLw3Kd5huN+11VRrH+Ft0mYf2tWmR5uhRFoZ4gWns8hngI4wrwDbK00BP4enZz9eHst5vTj9fB306urk8vzsdOkAsYNHoQwTqKYW7436C4C49//GnsZLlYx9HtXRngpApRyhkA0GB5J5b3WRolav6O9/Hiy+XF9cmn4Ork4+nlSXD6aeycnH+6vDg9vzGfXfz5+uTqb812p+cfr06+nJzffDgzHu85Q/8urk5/PT1vdHD0x+ux8z8frr58vdwZHM0I6LGMMphemKwiXPLgPkpWY/kSOC9CLguAFiX8LcbOrShln7Ej+0YrkZRR+bxlPA0sDhci1r1GTOSiDG95/fSoitBfPpyffj65vhnLRnJxJH89iDxaP3NPPQvV9dPJ5w/AB8HHD+efTj99uDlBpP9RRTlwzt71xderjyfB59Ozk2vYG55bLPMoK4sDQkBCTDewRyLgaz97dseObtNiCnrbmb9ubZldC551Glugmn0W6VOrn71dmCRpCTsgTb5pGJGsaL7f1BmJCX9Xb+2cLgqRP0A/zUK7ApAj1hDoWy4eIvG429hRsszFBlg1jN+2OGrobwYgEQAmT0qJcpD/YF/aZqMfvwXyT7tA/u9dGr3fMnyjtXjKAAbS5y298uM2Jq3X798G7X1zW/b3a2gam0golmkuwYxAxqzE2mEtjLrRQ2U4dhCGGE1olFKgpArzZ5BAqBupych/jMq7oKjW6+jJc/1ykwG0RnOfoZbiqfQQtL+qNlnhEeixAzIcKDo9HjthHKePQRIm089hXIiR853j/m8CaIsEbALAdOpW5Xr/fQd+LkA9LwXjI2fC1okW3V6epqWcB76+jUrvHVgVhXyG/3JRVnliqG+fVUtalVlVejMXOiER9z+6KN9zhjl2CM4cjA+Y4BTNlpEPb6PMYzwl2BcXl7AAaeZOaHj8vk9WDQL9y8mHTy4Acx/T/B7lRpkLWCKQf1WhOshviMI+TH0J9kGUQKcGD7hoJxTSUICeL0m4ERPTeiC8nQMHX4wcsGroEyyEY6qZ11dJyZVgrklzyRN5lcCkpeIlESsXbCLnW1RxCTzy4mJ7QAH/+NqaKVMPIIz8sEDOBKbBWWt0TTxpPV8JaLQ2hjOXDMeauUX0OxuPhTuHkWlApJaHSxHgW3MpuJecnhaesMZkaRUeTRD1IKyDnNQC1DgAhjdAOJd05GoftbtLr5GIYN7ifgFzGA2vRAGYufIZoObDPtgUnsFzz+EGZC2gC8BpjAMFYuYi+AAbuHNjwmQSeKAZ675+BOv9vImj5N4bOSCzG1TUzUbOdGqBrlhlbpEl2r6Rhozz24cvZ7ASYQLzl/tQzR/nhvPWIxAx3fmkAbY5V6AZ/Bd7zphZ5o3G5my3T1TPkeHxth2YHSFeT5E4QM0N9g3g9nrwQrD+iKD/OH911Y5mtMzB5YSUAUh4NPYccY5+PWYmwnHBNBWwvQyqMmOihxWjIQBd0YHwYAcuYfttwD+Kx7AtH6Kl2nPgVl2w3eCUd8IR67VY4maTrQ4W4fIe7CEnXJcid2Ahq8xBuZ2qGZN/ljw76zi8LchLI7u7WoWwWAzEL58zgdNy8bHi+yoPVqgTUbCBHzh1ZiDSQJzgOMgV9TdgDULd149gJ+DG1t99sDY2OMIP86bkZARAOqDUlRMHCpZ3uQhXKB6JND4a/Em1CeRzry0ZQXSIPM2Cvn6t953+PHKA0lL3RVoQAOOlQpGkFpJQgHQEsmWVa4e4DLNwEcWgpwBuHBWlZwdet1NDtMY4B9+3NYRcetQE3f1NCxIgByTQH9ydooSGizSNvfbS8n4DHe3V6wXbcQnGSXRbpVXhWTfYRmzSHMwu8J3DciqnZY436rJJZ2iLZFrHaVh+fxwA1E0F8g0dPKlc60Xta+TZICIVVUs2Rcr198canqRjwYvC7fy6nR1gkmwDlSS7AAEl1ZhcC4J8bZ3WCqm6ARFXlNESsLkFo6+82xTgH2GgZqVhhjn4q1sa7zDCY5gnQZrEzxowcEkvXN16cAQmQgNIHykajXpBLcCkvNuE+X0fGN1Am0FpBkIYDIk8iAUgjSZaDtZ14ekX2v4hqTV7cdG0fYIR6C/IK83kwW2eVhmadfh3xi/kQzAk3ThHSwiZ15Mt4Mm8TRmAH5UR+Grd5sab+eiV1TMjQQ1wkwmQeAJnUE/AN/AoRnPlE4BBCig3YyKFBxt/Hd227SSME4q/g/JxQkf50KjlD6iXs4kK2DnLO2chACcV+yDlo4MujtR3WgvxgKBaesIynkQBLFLCaebCgDBvbTjKpzoGBDYiEMAaFNs5ADbCkKfELCoctFBQ+NbGzkDMyrNgNK7tRXZdwFoEEea5za7uaNSxBbtYkKjmQRhIa/FcMk/4s2WP8IojgDJP4wPwREDaAa25h6Po7qwisDPywtFRMRjB7eKnZlvPZBMm0VqArpkjHp2J25sOISoZDfw2+ABmyABmcnvy5JWpVSUBw9Jszf4A0C0pUHdNSbM6RbjJYvRE8ave77iIwJ51vNG2wCPpey5jELSrQPdqRSb7uypyogo21vZePEvDHtxp2unwRDruyOatEZUL0rZt6jUHwFVOYTc1tfaCt8xg2/76w7QT48UN09igZEvmAiCCsBVt0wg0wwYkVR10UeMVZOLqfpIpEeJOaHXD3nbENAu2EVNxxV7U6tBuBzVwAmTsRA3IXopaY+UTTCk/4YNIBzO07Tv4aJ1yOEVKA3Sjps05NCXnjjJ8j0UmB6RVhsd8RrqSLX/cWeBF52H8DJq2UC1+uzi7sDbwydYs/Oc0Tn0z+XEJn28QMZHbO6Lv6WO6RQ8SbW6PWQegY1bYuzV6XJ/c3Jye/3rNM1Tf/CrDdfJeYMtMHApiqB0EW8eeI/Dc4jlZYhBmiYYAeOf4MQW9jh9WD/RqE4M6fsRPefhcVgmFeEqwZ9N8kYb5Cr8+Apss3NGrjGaRFVK0PBi1pMpjmdsat92WI80UAUV6rGELdr6pIUjNW1hs5JOXd+9a8oIdVQ5C4SfS1nLKWQzGzCoQWbq8w/cztF1qO6koRYaPDy2Se4FWgCgCDYXagUmzARSLQAbg9GNqHpD0FXKkLszmyNta673ACoGGIg4YYwgNvI7od5l7MFDkBkND1zYe+/M6nm+AB9mM8gGBiyexrIZhS3MIHTiX07EYUpP5SJ+fBLUHADND7tCGbaDeBcFw1ss1Ng8arZ3UqXYzGi1HW6Bq313xsnwwV5gGFE5Q6MoxyMvaArl24zvbZGxz9o+2wFM0bZBXfvC2TTMD/wFZt464DhCwE3kdTkc2xQ/RSUaSYZ4gATi2fxnFUuakGSjGh33JLOh4FCgMQda86mjqkOmqBMKsYzdicFVaTlJd5cC8aJqjwQ7qcA3bpULKe1KZUCw+fBCBAmpGPwGPlp1lR6iJFHdZpnleZbiJCKsWHIr+Cm9kDqWMt13GUG0JtvrSAVq3lwr2EYyzoBDAjysmloUuzr6iWz+cTITgOxRFsIkWBEcpYzT58qoATvP0o6uv1x9+PQmuT84+g21Qgcn8BF1BwB8dHv+gxzBSO1IdqMXdV8P7+NYda3TkEoI+AHA3WOfxGR3rO/DST56WgohvrGYGzcweV0CHfCVWhoL3jM9GT2SSIECnNQi8QsRrmVSBP+/uH1tpGmI8aOOT3glYNRB1gU6Hfc1IsQw1W1Y52JMlw4MmyB2WVlrr6a1B8K0dqgzX29cza8+p0fpNBkDbxSLcgHfiFSU7yL0FvXnEQTtkdmqxWmUoHpL4map1wICNo2VU1q43y2bcIIA8iq9GlF9TIVxhYjyOMWjiATwmBlMiIObGDBm2zIkHzDe7g5NK3wLOePNWcBgKtQGD52+daBeUfr4LKI69omxsQalfyA2ot8iiimLJeUEGxhkIJV771vZosIUkFrLFVLmbM7aoSFGGVZluwBpZOvRsn1qy8QL6IZHGCicW6hqTNlsoXrei2Gy7kyjWdGNQ6AmBe2rI4iiB9QdXuNOotUPSijXUrJlaouQUWrxm8mliTx010luWtN4MLOm5zne9NseBrgVgXI4dOQfcmzIioxa+gf4YdDwQXb8TRalejbo0Qid0yTPsx4zEHn4lTOYdIDqSJBKPh/Jlwo9CC2qMmTTPkW3WrpoOPnJYsNbJMxrotcUkDZe0OY5vWBVWTq42hB2vzxJNkVFjWbh/C910gaHIoi/5xwlAOY+62mj3yaAVhsWOQHvADyxOTyIIDiTFCJDmk7ED/zumqMLxyMeWIyvuvZPXw5hzkw91mVNzVeTbfY5U7DwhwgoXdKxifYnze5S1mMKPNpwpH7c98b6GNs5V4+XhI+0Jc9ye5kYCmYIo0BWjIZQn9Ua9XRQdVQrZLEVQXjI6TpJ2KvHqgBgoohWow7LAvcj7rI7m2P6RPT/tVjzoYYyyh634qrIMllW96W56PcPG83oSMpudgs0JS6/i7hbMWUD6YQZOw8p7cYk3MNXKQktt+YljEQ7jbX6L3oET6650LFw8Uby7HbgMIjRwG9XRAtBj+IpRrp+9jnpsfm4ZUEFItAy1a8EU6unUqTulLqtoWXoghgtWT2TtNYftqVr1BiGPbbmM8U7OIxkVU9Vd5YceBdYQE2Rc06mOGYF7QB4rlZ/sNgLbjtOumw9TwgWfYtVPDxVVjGOuInJ9xQ9ETlkBwSatzIM3rZGmw9m0orJcyPASm37S4aDPHTtKljIp86bTl3u1zZudnVpq32zs8aBjh4zJae3CON85R6PtfssL1lJlQhUSqAopENRAXXo12mU9ed8Dg1EAjaQtRS3ABwalZRaN1SOQrHfnO3GkIRlw8yogrNlfW6vQqNoy17LrXsjVLDs+5qDb+CZH8Q0u4O6mrhFiyEsG6LXX3jZ1wxXaPnWj8XDURg3SDKkOuRpUM6Ly/WuUL9L9iNOiGGGNA+acHDdB8YBva6cTW7QU020eriLg6oIKinz8ymUiZIFrGdAtJeKmBqntBWU1fFXUYsP/VuFNY2OFWI1YzyZyQdMWOCncIN2p6u6t+T5GZcIR+A4L0dAqaU9Tt+frLYJl7RjVA84vwM12s4pqeJJKWM3BRmWOWa/QKe0zxq1LhNrrQQQ3iE2MFwCpYD6p1+w3GjADa4rpPr1tF7kI723kUUAGt6YdguIkCwjickdOSS851dvhAZ2ICvNUEYmDJRgtbpDFEVOFHxa1hCAPRv4yTjvuNFWhQOPeApUm09j99tZGH+28xyWK1h1esw7XHepBHNbybtskIv8Eq0PrgcLk2ZP0+EOXIKM+g2KXhMzc+edUDboVDCez5s53U+eoLeJRgfX10Wko8Fd/cd532Wt7x9o2b7bAPEU/pj3yidQJmvcN3aKM5saLho78ri994t7G6QJMSdW/Rqmb45sPgGkyLQCiv5gn5TXvLCTmzOSn1yHdiKQb0oyNYFnTmhqQWwNKbq/fWQHHgMOxiiYsFeR+sSjCvtVoc2GPAfPd1GIkzqSFOJ8dznv43rp07TGNjEw7azt/48hvt5PMJfZsI9X263zsDFnRnQ3cTDbTvtVheEyCBMrvnA/t5hYUvYnfveti++/dqz3QGrmpSV9mqmMm9m1X5FVVdaj3QC2YB7rx4bpEsLfPtRcMh4qWahiWaiY9UkB1TKqa6bVnuJZUDMq0DONvkpsd8bSruu0NN8iyCgwXaF/fHXRjW1zbYEqAgxJqeAv/abo7J+ug2YAy/JNz2Bp1J73b7x+6rWPlSlwKINYC/Nk7GM4Jl2WFR86HrQk2jaJCtDKXowEP8k2qwuJHTkmSbMlIqH9IuD6/swfQlsAb2uUY1qn9DRYaS4ptg1kpZ4tRzEKrIHf0L4riNZG2x2d9i/vc7Gtdpzo39gad3jvitCX5OzVN89GAxYJYrJjE4EeCgYq0Cp36RIQmMS70PyqqchzI3+JpCAxPYxSFCnnoQQaeQ8jrQAO6ffKki7zSOYNUeEPkTKUC7XGAVqhMN+7D11YFNVcXP+wY/6FCrQBPug7FJtjgGhKWXdoRc1AADYWXB1r3+6NerUYZdZUf5QX//nBfbj6dMsV96MjSMtGTPzUm1Ee3nuI0K+msy0l7yAw1lPnzpFO1vXP9F1LKeoGFm0VJAqT4alSJSuirgTJxFTjmKtyscjGWYpyHgh0SPoRRjMdFaJsgAFAQKNWWSxHjoQZMYxVOleiGxnhq+lip6vUGxLv4ccQJ5O89YYYlrC6xEb/AcGiBb14OJ1g9XGDYXvB5c/d1zyb9ib+4t/Hf2f7R3L/PKP2cCSLv7BjYj4vaCE1Oaqzoug8u0wed2KrOZoHLmQOu7NQFnVQAyofp7AF/pGp/3V6PeUmVS4Us2MzilKw7XfIo8gUSrLfMkfUXZwokbpgvwHgCH7cDvqcKW/EEVAvSewXqtbmwrC89qRamlhokLMBRhBkN1bzpwtDO1rIbcVqh7nFiHiukmsbHZG+bGQiN961wdMEVqm6R52m+DdgaWB+Qt7Si7tRo7b7gqU6PnmClEhI7CF7BLKdHr3V3sqW40hoFUPzcKOPjTdX2l29BcpSwkjJh48pVgWWk8xO7FOJty68Ziw72kSXRZqYqZNmFtKa4JltXV7iWaKMuO2nMRA03JsByMrb4osrFtshyFxYEjF9TJTnujZ5opzT9qI2hFGzOgdSUh/JYyyYFcZHm9dGWzSbEG3PofDtvazmmuuWI6kSZ7ZIHlhpp4cPnCMQYIIoyM7j4/Pns9PwE92QJu8JVzz9enH8+/TX4dHpVb2CUKzKFt69XsKlG3S+XZ9y103MTlihIQAmRSLr4chmcf/0S3Pzl6uTDp2vZtlsi1wL/17Pdejnu5W83f7k4/3r+56+fP59cnXzCKR5J4T2cNSGmFeE9XtoToq6fohjUGpaiPI26S3Bib12Mu4IB6j7ZLrnAjQ6NGtt8ySFv44qKS4Kgl7YoV2lVTqEffYZNPDUaX998uvh6g2M9TOH/Y55UkIhH2F4F6na+v6IuGr0DGaIG9rMUA1zIzt0d27AhtESGiWKEntjKv2QwngYXrSwum5zghippfHl8OErWKVhH+NIsnJnhACDP8Y+/BFxXOeUSlxUYKg+CJzOnjCwCw9KPKkFp4o26I+PywdCb8MmTK1kU1l2NWPxJq0Vob1TqvnOO37077smzaNZw1VJeXV+DFbCJShLzYmXKa13hFWNChUkI+xB934DJAhTRBo7zi/Pj0fGu4xfPlMatezNAZy3iGPzEOH0kaF+iPzfxkcpIYnOeXlfLO7msllgY1gO3aDdYD92iqq6jtgUn1EwwqrVPl8EN0RFXjbp0QVkZF/+B4LuP4ji7NRl2LO96869Pf705ufoyshTkEYkkVc7S9L7KTpr6epBIgzgpTB7DqPRw1rjZf+xFwtj6N9z45ClD8/sbZ/zX07Oz7mDd5BitchELkXmH/nHdQ17t15hF65IcMO+wtn/FlueKzx+lWcDLh2kB+jDeNZ5pC2JydT9esYVl9qqggy7cUTt5wuLggLdTqzs2UtEPPn0F/pqBCsy51cNk54mdycdOQ5RM7ALm1TixCm6lSErPKI8fugOQ6//r5/pkLdMffICxPJ+IAH18UJfWjfkh18X64QIeV9q1Vw4SHzaTpXkBhTGCwKjP8xnhYnY0pyuMqP2mKkp5wHTNPjQHA/hmFEqXgmBqHe1Eawpw9ckXKLy6yh0fmnfEjMn6cBhvHmsBNhgQShSYjAFVYYKn6SOMZk1he9hWCSHV9dUoIOHaLdCT2bPfrNPCTdUc8hnXnvnTYkjLRQdh6aHBUt0XkovwxdgItZLgLDQGKkXdGvMICwh4YDN0jE/fj6Xr4qiHCtZClI9CJM4RIfLeNg155MB09o3nQMkVuK0bjIegKKAqTAcaqhtHP3799EEfW8ArDvSphlhRbfuZa+Z4emjBUHbrOdJMTegA7n/0NHMLE/sp5gYu/4kDzPW5fFnoiqa08o34dIs5bvdSy1FzVnwUugGUxAQIiQlXTHscrTDhYswJ2cn1fZe2HFiC/YBak6/nVkuDep/tq11L5ZAH9ZCOvoegRQigQfOiVo9FaQsXddCWrydoHO+uL1msz3V3nIEm2aKCTj8kS0rPEMwx+cSjcWOGEud6qiAKqexOzWLgjgqD4cYac75NwryI4v81XKhvvSUjonUDriS87KIieYZDo68xmNpvB7SIWHW3MXGZbDyzXtGHKm1dgfVsxHv1pgodPCueyGEdfRGgXAVS9FS6iXgoF1gWpcsLAhohMQ5iTwxx3A6otfJ7E4vkxqM/dNhVSQ0z6GeI4C5ofaLW1EPjtpXT1kcdOE1zp62l6minDspO6kWHRdCly224lstDJq39iDHAllp4VSaAv7kHw8CTxgrxD3jMMuoobzrUlzXUV/y29/3YdupedqTLo/1lmj2zxLAez6/vEm7d//Cv3BSx+106iiv0NTqmBvrXr8+Z7HCNDd0yQHfQgLFu3A9k6iomGudPeFfpjeqajofK/gyeZ+6ekYU/+/KyFXk4VsbNZGyWTFXyRFvBthnebM7v0cHmC0P7LOMdKrLBXmbI6u5RO36jeTO817jgQV00OXz+t30nhMFfDUAN5qKiz8ZbbZvTxXQvxi0SMjD9qm7PodsPKdaoINBNtXdiExoXnx2jhqjyjLMI2nrdZ1moGZRFNl+4l4WoO/bxDnq+g2OvcxsYXm0EIiAsg6rEO8XUVfJ+kj566jZ5H96NwBJN+RY7skRborMDWckuw+HSd5waUkzms/vuqKwNSi3L7GBaLy2Q5FV/Yb4aQMR8bYEhhYMFl6GrLjtgOlePTnrOLjJx5ztdVqpuj5ro/U/JIrlB6QCS/GwR/hbOMNaNnA4G4soELVVmIdNRzr/NeElq5TY66YGeIR8xQr7DulgpMnpzYy6GiifmPbydQDJt+tFOlzvIDbdVFlJ6PjAlnP48M6I0ZKSzn2m8N0M2cxUw7s81062SSgj9+y61Mi+2QhufYoyhzJOi1wNwVb1+66ort3FzxBA9bbVS2y9d2xmvbZew9RSUmPtRmySjnW9b23JrrqtvXLM4G6qMe1XlVIKkt2Cr2psn5tgx4pc91wGbk7M7X9y9lovufEST77zYNlHtKyt03zpNi+H0LUaSEQAu02Uf2TTHqWZu/1nvrURUMNTheKZg/XSYcvKMF5f81babxn8H8m27mbtbsqIdvB4Pb2zE3aQvRiFpPIUsfbJdELOR0OaZSIL1Kfu6nMOI1nbv1raNZpZsHCiWlBZge9AOpxtWRYedG+NKRbFFbTWqI2TiwfsbSmpKe4BTcE0fRpbKha6SIRXYg0PjaG5QVzGgPcwVDEoBophuwbZowL5qiWjdQUxFd2tVb4FXLyMXKXQU9joqDyR3HMjiPlfZ6/Vl+s0UrIbO9W/FAGjZwl8WD9Ii2Nty848Ernwb+XX0LVTo5Kx6Ew68izWL1lMdYJ6x81fxLD/dPGeil6N2WuCBlatVYC9/7XCj0hBduwU1RbWUiX0bwS2zmVIlS6NWyOoAtUp8sxyvqDB+guPFGl2qvSw9tHo0N68D7CDWpwn6zHDrLYCKdGRCdRuMncNeb9ly92ADWvv1ICzL/YYNYJ33Y2c2HwBn4TDpfSL1JQd0N9hr93dRmrU+VLAtu9OqH6nqH6yDA0AP5mW18L3ayIO1VH6IYQpsNDuazCUPPSiDneHhEy0RsF5awyBVc6xyTPxsdsiFmzpc0fl9FeOyXSO0QKGRGsrRvBm8kgyMv5aSc4qSPvsfZI9LeuMpuYiVLEGwSpdBYPakS5jUIJ5MQcLCYQ3c9JJ+3AQoF4IQne6QxByELDdWA7aKZhsSvqc3RhdQJ6QRrOvUM/NKZoJl1ILZb4rdiTibuuhIqTJlMt/ZKp0Y+aeIw2ptu9cIbKomGoh2TdxhWtNYxrTMH4RTZLf8cNy2OUVPeiI/E2J3wNdpDmZC7Ayk3YjI9cSHcTdzWgrV7g+2bcHUlmbq+k8/qwGAyinNB+Zyi1EHeAECB1h7n+rNVU5mGHMZ0W8zikSpnUilDOrPeLRD/XQIOd+LtLzDLKCqLd7IOo56yE2FdUl4jf/TMq6w+IqPpXsWnsfeHTw5jDfGwz24eTGCkOMv0FF9ISOLQQV5krj+FUrztIvUFj+DCMWqqVu6NMkwygaH5+u4BobXRzZxTZjrdJG/5O3BlWCFrTIyUjAAWjVH/TDcX6Zd3t5zEz7tq7yM2dvKEnXNkFPXVi2qFSg9YA9Y3BV5T/gTMgfmUQR3Kwp5Uexjrsc2gZ8ODw81Blydo8qE9jHNRpVyxABIhuIujZWrgqqy5kX6gyMXhpqyBZ60KjKraPa+xQod2256veTrYM/T8jOyaq+5yibZ9jJs/vnPKWprLuscteeCaj/CazS5LynhIEAjIAikEubjcNdU+3fyFJUemQigW/8P1BEiBQ=="
 },
 "scripts/train_pose_sampling.py": {
  "sha256": "067759315b99af7d6099cbe67a1fd64e492c4b41dbdd5dbc96b50c4a3ffb660e",
  "size_bytes": 9892,
  "zlib_base64": "eNqtWltvG7sRfvevYLcvq2AlO+npQaFCBYIcBwiaxoHtHKAQhAW9S1ms99Yl17FOkP/eGd73JjtBA8TSkpwLhzPfzHD15z+dd6I9v+PVOaseSXOUh7r6y1kURdddReqKkT1/YvnyrsvvmSTXb/62LOgdK0hWV7KtC1K3pKGt5LRY7lsYI4KWTcGrewKzbAWMzs542dQtzBzFGfxf5bAs/dpyydK7o2RZnTOyIbdtx9xS2t4DV8HOgGdJciqZ5CUjZtY+JwT//gFaWjr9UfC7VckkhXXUzvxH1JXm1lB5gBWW2Wd4dBoeOsmLM72OPdKiA0lpUwsn+svH2+u3H/99++HdTfr75fXNh6tPCZioYKk40Dd//TUhTcv2Bb8/yBTFCyatVGWkVBkpdUYybOPfLt+/Bd7p5+ur26t3Vx8Tcn318fImAfUU3XskuzFUCWFPDWth78CK5/CXy2NyRl7wr6hpnjZtLeusLhIi6q7NmOdBYM8czZsKCX/hGcTsj2nbVQuzDzBGmh1Y9tDUvLIWJy3LeBMyki3lFeiaKncJJvTBew5pSSu+Z8LYCeTea5OnloWV8a+3nz68v7y5TcwiY1+hCdVqTZjVJZibw4k7j2Eia3kj6zYhZV1x+JJ+rdsHBs+wN/fdbJdXTSdTPFZhFfb+Y9ZoSbTKlb28Hf7b8RYc+ez66uoW3BrdK04VrzRdrFom6uKRxYsVaAgmEdvXO1icsz3ROsQQWHt+v1ir42zrWlomemIb4Vi0C1jplVpwrCk2BMUnJNJMNZ+c7/esFUR79xPLwNkxThlsBS1yjDQrDG32hHJ73qJYJ8SqYYdBFUWV11mHDglkhsE2smPRrqejZeG9ONqhzhNeHRteXi7CCsj0e/OxZGjsRqPFpFTallpc5E4vIvDNSXDDzjX1erub6RX9+IvuannoY6GwmhB5YLBXmknCACn4XcEQWIl3JhUzQ/1pUdg9PLCj0ggRChAaMBhGEvPIq2E0xl5zN4MeBH5dinixGKhu7FoBScszWhh2M2YNwM+ZGMJWgd1XhkAoQFbffnY+QAHNAcxI7Cz/A5QYypRdWwURayQmztKbKbz0njvpSgsTgMA2hsQjgthLSN1JwAJwapxZ4ZgPvEQP6iUregfDnXwmIJdLNVB2QhqH3R+NQ/xkQCod7NgPx2IFypgNsCcuJPiDCoZgnItUHEsw5ANuOWqxMNBWUbu4Y4QCpjBxgNMCVwkUd8YznKYhqy+qZQWV/BHgv1Yb9Ooo4w9WaLoZteBRgI1NlhuGE5pNHlpGc4Hn84uSo0ZL+pQKqEsqmPkHuehPtEKkJZQPMDGIm05gLHYtsVyRTh0lKEtw76gGKXjJpZjUpsWguKMS4kL0I8Z4R11FOwxik7d7BCp27NRSTRFTtk1HrpKZs0cOSik4bLoI67lgHMydQ/CXtFBHL1gB50tgIa7D6vDdl9/eAuLk7AnRu+AZl8Vxdm+nue+7AutKl7sNPdhRi9FEA+bjim8FCVqApeKoK6AoKI6SZ7B13OJk9RY1vKpYTr741ZZ7boS5mg5ceVTfmTA0JBbzNljP9lV1lNuohJq3cJj3E+BIghKsf7rGyzbD2k4H04mDwRDTNFuVArB4ky3TpWCn3Wv2iLKC0aofaC5/JD4zb16YmpLTtYDB4oI5fAQ8B+bfdHG0hgKx1fsFnTUomEGPF2rfMOqNMKygo354hUvtIPDBcmId1hIYhZhY1l7NEWfjyuswGlAnDRxOlH6EiQCR7GQwNGIf4FS43Ax5L3IJek0C53RZe8R3ou5Z/1BZBDxc3agPxOUnbMui3WoE/3hWQaG4Jr6VePXK+dj3IN+sygdIRLEpsINA7LcNNm4xHZ8/twubWs4BBJFLvkQ2NuRU27jK6uaI1VA8vRaefQfjltghryGWABvVr64w3Ys4Hi9eoWOkaDgo3xSlLhmBcKbxig3rJHTVPhwYFlwoKPgE8I7unWWskQCPWKGKDs7hkSO6+piG9SUXAnhHYThuo74mAnHOqKmW+b4qMBhk8aWmX+FMZIs1zdllQeDUb+PiLd4s6HkKBXWinGuu+Vo83yxDoaY5R0kAHFMqLpx39JUFR78HiSJFz3Y1EHKwhlnaJZpRj6rvAj1evcMnfN+X5Ms4MDQDSDTwDRi1p2CIPPpuTj1j0BEjar56ZTlgjgdcK2lqkiiQvRkZK2q6Fttf5NkvOCJUJ0grSgUIbcylyz2voH6vACFHDDPYEjhZSmXayQxBxVzxrKr6a2xveVYwB12LqKHjKalURcMYfkfMZ9DjRKLtwWE/V4/Zw0oGSJN5vJhmM5ic4KQlZbTNTygSTu8SMoqytYmxMXuDABNqhj3cGGzGnBDP0iMtC8/CMN9Gj1Aa0ArTRwCl+DAimrCAKSDWppJQ0G8L37UP/x70T3hTcKCqYtXkUQU+NO2h4JrgCNB3ZYcJB1LXQdh3qKsfjn66l8xCD3B+T4ELKFvU9+i77rIpHsT8ChdYtFjoOFQ3bmTjN4fFPpcpXovq0lA3IMG8kHUD2lAEjR3iL2K1Yibb49qpH14cOKSBdiB+YMfFqesDZ1p/Q+BvkvRdwMxdS7/cDUSO3FQpYDzVMZ/NMWMp6n5SleNaCljFdPumwXW2dbeZsbnTHNwG+CpNVbYFk2yDCdA7yXhvihOU6iWkd+iOGav0UXltzBQMZPAMqLsbpRw8VcPIlJOTnOycVS5XYe+uliDAdLN+J8BujKCTHokiCszVc/p1T5XheU1KRJXG5e9kFlVdrz8V0zyoDlMRLy3bKOhXiY5Guxm3xO8AA/U5xesGUgXeG8GBs0YMO6teE41ZIcOcMyKaLg4882XXqFu6yeZaud3oFjm2BYKByQmPMm3bTN+WWGxUL1xMCQ1NIK2gviR516Ir+P2N+YcQP1XrL7TTzWW9/q2ckfuMGFNZK+A/t0nH1ExDaaPk6PZrp35M6OlyXol3Gcsknd3U2vlcRaL+ne1JPVWptX0ukezsazBd62LhTeLfEZUv2xZfXVzdmC//ZEfz7fbY6NkFoYIw/Lbu4x8wVflpRp1U65JRdJxUMVCKYNGrnmzpbcJFBzQZR/NkYnWzvVynPxEonC4hTgxknRkAQ2BQlOvxVjy5K3MtmRfWR/UJLr5wUJwmE/mey3MTNecFBYduZBQ0AI5D0Hf2ZABSdYUUpwWYRatMPPp6Yb5rcg2FkWJ7J/O4eLkpegWElzfxus5BmgvjRY/QvzmMw5v0Qb5VOTgA+f+D07/IL04cvY/92WDoG+dnDsPJp9xEZgMQLmPV8+Vd2Yj429zN1eh2KWzI3Ibs0G6u1/WN4chas3X3HKtxxeC46vJvvCAhF7Nt+CiRD7gNp0/ymjpZ028C37mQ+A6OWhT117Si1Ubh56L3DurCx5NCutfmJVIJh48Xq48bLMrNqyR4VtkIYbOA7jzGywpctH293hncfLSFvOaHIy5uC6Z4ah4qeb2xLyX02PZCF1ju0iIEN6WvuSYJ7hTU5Yjn8HrXv1Uylxj4K4xWv/5S31dvDcVnNRNbCAPv2KQppN00DSlXNIckZ0hi8/YLzktCBG9QgwRAcE8B7Db4guwkqbs59DTDn02cpDfh1BNuL/oDtJ5VvMCKNTvUHCr6jfl5xsvpzZXvgCQhB1Y0m8i/R/k7eff5C/oCtEEll3j1hh0aJf1S2ZQXpf7xTCCz7GQHnntMAUuLTuBV6n1bd008oSpSjxTV7MH9Vd+8wW6zxVdNHYtOUoHjn6KaMYsuIuwduzkbiFV/xr+cpjdX5j9OWdKnpb1TD6lffqTIoRViidfsU/J/vbi4WFgAEP6Y1AdyEkFUT/Xv9m23fi39U2VhMvGbqNVnmj1Ae/qplu/rrspnU6nOSfvoG+7NJMBVCpBYsjT9vibf1ND3SP8EaoOoJmQOg4vhJhAeAcgsrQKrNEWwTFMDVioVkpsjQHp5+cRlrKAUcOh/oRfVNQ=="
 },
 "scripts/partial_front_sampling.py": {
  "sha256": "9f9bb65b0db543889d52ce1c6b8f4c87fbdf5094d784bd955d6f06d89a664d60",
  "size_bytes": 30180,
  "zlib_base64": "eNrVPWmP20aW3/tXcLiYhRSz1aftjmY0gDdxMAE8SRB7sLujFQhKoro5TZFaknK7Y/R/33fVyaJa9swusEHgFllVr169encd/Jffne3b5mxZVGd59THaPXZ3dXV1EsfxD8WnfB39enlzWmbLvIx2WdMVWXm6aeqqi/JPu7rdNzn+yJtim8O7rFpHTd7tmypfn+6yoolWd/nqvp0AtJOTYrurG6jU3AKkNj8BONtoVZdlvuqKumojqfBdva+6vOHyddblHUBXheo5ifDf3+oqV3DvsvauLJbq8e9tXanf26y7U7/rlgHv4B1UV3B/sao0MI56q57ax1b97PLtblOUusvfCn48gTqTNVAlfWiKLk+Xj12+qtd5NIs+NHsopx7zj1m5B/RTIJweT9rkZdYVH/MUISUR/pu2d9nly1cKz9V9dpunt7s91G33ZacJ9eNP37/9j/SnN395mzChU6R5omiaZs0WH3aP6TbfLnMoAcBNvk7bvOuK6rZNoo8wdZtHqLm6AxykR0AvJXi7uoBZVZi2GZK9yVfFLk+LNUx40T1KE4BaQI/UdFOX67xRzf7tzfu373786W36/s9vYFAJjniTN3m1ytMt0KhkCDC0rm54mEW125tRNnm2TnE2se//3hcN0PPk+7c/vPnruw/pL7/+/OHn735+B5SODSO2Zw6vngLmuxIGfHp5fvnq9OLi9PL6tLn8doJQ4xMFQzBEUJfZ8vz85fLb9avr5Xp9dXOTvb7ILqDxenN9c/Py4uXl61eXy/xi8+23l6v1t+evzl+vX0Lb7Pz61fWrm0188uvP796+B1CjeAUYNHUZJ1G8AsYqkIPj8cl3P//5518/QI15XOUP6Tn8d3GJlfTTjf10eWU/XV3YT9fXcXIS6f/M+5d2rW+vraeLi1f20+VNvACqrvNNtC5uYSqI4CNk2Hw8Jdgs10rKJsyiI6w2We+3u5YrA4vBnKX3+WM7Q9aH5xzmIoO5bWejOMFOp/HYRvfAf1lZ1g9plVWzH7KyzccT4BpgmdF4PLnLPzGmo7EgXtbAJ7um7mrg/1FT111CQj7zeUUNCGoA+VHwqfZ4AjxYlx8BPJVjW1WOv523+MfUj4oNvynaNFvCy32Hb3PA2RNwwavtGoY5FtoSX48Y+IxRO4t8xIF2ipMjNVBGaF1sQKba2AVn6RLuDWF73A4wN6Tn+5BdoOt6tSclPzMSadFF9amqzeMWVMg2S0HDtKDd4wV2fuHMO5oLU/++qNZcK15m9TYrTgeEWCEYHwBmdAFoKoE6AA4UAXLlA7y9NVQwAOKhAWppTrdZVWyAF6UjeJed6dL2jDts8o9F/mBU0BX0fHOmmoouGh4R2WBU6aBfm3RZd3epwTErU1AxeSsIGD3jClqMzSKqGW33bRftgUG7uzzKVqt81wEX5GVxWyzLHM1+xGYfuhQSKJ1PLOBYAYsquiReuIQDYTbV7gnRjxGMJboHMxQVlQY/ARO6bUHIUUFB9aZYZWXUNVlR4cxwBwMsb/CAyjjbliETUWAa9cxSnO/q1d3F66tIWha/gRFze0FslTgnxCIrJBrgPhrxrLvTmVjz19b7ZmVYRSPzvC58huXsTvrlph9Re0MKIqin1LsxqQ41YNRDNBrDEUKnaRRHL0wrTTScUSKTQVWxXMpy4Y4DXq3RQ4gX81hNPLgFH6Gwbh6hbp9m4car+g7sEYz+KwaPSM9jVHFIPSQAv1EU7SNh1LPCwSWMA5Fai08IChKEyh6Ceg12WRERnCxQ1UTGeAdynwElYCrWRXZb1W1XrOKvGaXpai4dvIhi46Ph0HvjRF443EyzHdqXpv4tr+yh6mqefLXAXyhX+WrfI4kirccoqq7QSQ0bIUGbPBflf/2K1Cm/ZqZIgYK3qKhEa7JD1tPAAgl7z9sUFG5KikLs2dVrq1L+CXgVXJC8RZVMMQ01sPsbsoMMgdo2MEEQaHSP3ALRK1oKJEI17/JyncJ7sAarcr8GgfJbeTbAM/drDEaIu+qOAzEIAMpi9TigYTXF56wxhXpX54ScVYreEam8rkE9vMw6UMVS+7o3fhusqBm3ycX5y/PBmnrICPvi4vwo8PUOQkjU8mnb5Ttp/frGr49hT9pASFrgX6bC2lD3UFfZvgNHBiQT/dJtzXyKDcmd1a6XFUEv9+vbvOuRnlzvz7Hi+3iqRQB1MYvaNODakaqZko/4JG6yTIq23OT6st6gQIzULQdyq3oLPlMBlkzFYl7jKGujZQY1+f1RbjXHsFDDasgVtaKrMoz3Uc21q6bYmXAuJU8qVbw72ZECVJUszN0aAwGAwWYeo0JstdKaIwbADtGz2hMrjp1ZYohCbMcNFYJjPAiaIiE/TAdXLFz4CkdO0SMMbV/dV/VDZaQVK/QdcgE5NwyycBkHvLHpsHcMHRHcKYFHthEPO9WspXsIG7+YJtO4fnZnlkOYuBU19F7lgV60DwxtbCdXlWinaBo94zOF4Pou0/Q4x8qDpaYqRT1tgbLC6pBRG2v5VO21uyOsJlMw+6muFNvEcfwrCxd58fmnou3I+wDtsUJWJTdee840zPYPUZVDUEZRXLSGn2W9Iz5a1XWzLioMWShhd4w0C1KGB9GU9CPxozlW/Eo7xpRgOORfM1g9IxKbS4PDHMDYqwKnQweeDPMuq2D+oNrnpn6A4EJch3iBMvPAgQH8BdHVMKFfbhUvnnhsecnxwpdB4VpMGA2LDR5mjxZWZLKCCcTWmIPJ1yNuQ0IFBS3K333+OCuz7XKdkTM6FZeUojWjFYuNAAMOBWbEuPZ3EFYSI8VTh99xIovKMoNqUIkQjYNFgsUqQKgyATs3skuMm8lx5yzyVa4zMROwCWT/GJ49y9QeAOI4uLQfLgzlZhSnxYlGW4EzCOrIbzbUu4k1NQ7cpQNU1zrotlNVPwBzRZwp5sYaMg8G69um3u+QZyTBPjJFyD6gdMj2EKwJiQOK6Gg8IR7AYgjK3dnfFOB9Ikgs5XojN1xQAyrzasS1aSQXF+QrYU6A386LBZv8c1TnlAg9n/Bv+DPmKJJqvE7AFxwPhM1gMfXk6KTGx6ItlkWJDgslEFHhulgyaebtfjvagPbqDFLj6E/Reb/3RfTC9uJZHCfZbpdX69FnI9lTSx5UCkAEWlkGLQP268VgVkA4Rzw7NVnCxl0tWjpr0REqPo3G2jjqDjUXDXbBmgq91anWWnM1kMXcKkeTXi//Dk8t1gX6MSknlA2mJM5gL+B51+BjNhV44OTYIQiZiXMCXOVD5RcHCNQ9DIK9XDypVBYpT08N2xqYp5QlE6OxIj+uBSlPrKTIQmHFOQOSIIs0h1LSoAYV/Be23gfnDdQtxCllfputHu1w9VDGyIGhY0KTRDFQRKHVSwok1xKmftkgLU6gzKPJ6lzeqMzOELa0WohAAoywiP7kR8lHt3wh9O9xDwG9OQT0KvomGn0N5DGCdqfdTUj0qKwSDkmkM1yndVU+RlKh2UMoUNXgTIHtaKJbEO+WvLx1vipxFS+qSGFz5KscrKpuWpe3CtS4pxH+1NynXjAG3iIEaupC1DT+VkDGdsIDC4Rr5X0vDsYqVo8Ymb/UBYwpvb25orcV2F4Lp+hfPZQH3GwhV8JjtxIZQKBdvduXnHEK5zLQ/BzNxgekwBIVKJSxeQjX26JtCRVCtGijet+1EBWanDt3BIhrj313B060wnqFZpt80FhmfUq05IdxTxN7WqiHfk81x07WaepMchJ9881njHYNbA6S+2A9qG5EH5CroKZPQnp8/PTkzB9TxFO1V69vrnsOkqYoV2VaelwhaTpD5qOGO8AEwsZhZfNPJYlHEWcMyCtBM3t1NWBfb14mYQt68frVUzCTKPrKcrKkd6Zu7KV9jHOEUOFfXBf3E7LAexDKKl2QOJwpFYQxezysxd+prd8mRhLBQ6pLpxZLbg+kFgj+QQjvmSr84+nEziWZZSYn6ufi4VyRRgu8QXDOdfiPS+ig/6YuGYk+us58cFBqEY62tEx+pT8jAjhp7/abDYY9zyDW7pdtV3SU02x9vJKIssBeHqt73OUjLkBNV8jeoIvojzOujz+uzu0FbX5Nq5LLHP7vHvK8ijhMuDp3DFwSaVflGaIlkUWiftqfw/IuI1CML1jGizF4Aaj7uIGbg2UM5iNuBWFWtc4/jaPf21ZtYbx+jhywTmLegdTntLAJPprqRKV/dmVWodOntlcNJ3++KPWiyYDOwTMZpuPTi7SQDCL9ZHIQdgJzagfMNBo7+vys6DFVAZZFH2varPl6ciNWnjCo3WB8PboYWBiyFipewOS6EewWYjZkv9kwm4fSdWptSbE/WwEYu7v8zjG/0KdHjMl+h7VGggNlREy4qAP4I4ljaEPzMsd/F663gKbM9G5CNEzL8lLLlNZZhmMrXM7idSxMaa66kXiaBqpavD8Q+LEN9hbKlo9pWa+yLgNPSIwPovP5UKyD22hYAL2x+RHMAiQ6bMM9Ujp0HLDcIs7KgBtR/kdNuXFvVNpeYWeCdwtha4lVGHyKOjXsdrKChBrDGvMQn49ljaCVRYIW9RXEIW0b/cJrMz/gIN5LQ52e/vcm21HUwqll0D+8ktrIXlHKWIJ+gdmz/V5ZLCkkE80KCbRjmuLmjDQFC1ZukshRW86CytcuqkiedjPRQOmJRdvtya3/5QpWN0UKh7VsaCbctjzxJg+sCzLyfXmtGorPvWZfqo/dMQ4pZarF63o8yymzyCxCu2WmsajAfELsxbOINitvrImjdUjwP5qsfOyKVTvBhOxkuS/Ktd76Wm2QE/LvoeQdtbdycVQDpHJHC5TVrscR3OMEqqQPdXOfN+Qq8/KylJFCTNviN1Lp13bRuql3KYyrMyu5B3aCSJ8cu8vaO7wvc9zHtspbXEGFrk439b6RHixexJFDHIwOn3TOL8xoOZFldilOWtxrQ/tFcDkE41FuMim2lOZu+9lmO+TXWxpU7gDe4e8DHDC2chiC53Dsbfdb7Sa0K0wQFJrfZbucJiT0fn4e2jEVW8Tt7+AzcwB/sxLVTgceA+DXMPEjhm1hSDoebVJByzVGNKjgea8OKWesvi0SSQS6BwikZ1S/RwofIUrkW9sl40maojFK0xPdnCu/FTF/zxVdz4f1KUy9qFPPMbIMEbDESaApiF8z3FYrfIlVHIwDPRlOVC177OgU4Avx+65eH/A2wAT6poeCDIxXuyjPYPaN2cG5jGrc9tRHEUlMslbgwgBO/sLwAc6+6qefnkUHj3Q8Tf0EfXzWzCMCOh4iPFJ4ZLGi5b8o74NBj838CFetyrrNR7bIEcS+3lTSN5O/SWRU36ynDPEIAUWNswHF1//P0rIziPhkEmYeb+JCBTjBeFIB1Ivq2Lw5pic894ArcJtKtTdvkohxYAfC1HDfHtPLbV7ltKFdgdAvwIVSpkE2rBuTR/GiTDqbPTtudjZYsclWyVdt5McYvRwXRrdR1uSUUaXl2dt9vXeU28Z2HlzRpQLkkfZOsDVh8UEPg/6a8Vo8zsPV9iIw5AC88PDVSHH7QlbpDGrRtUaIeXueQXYownMNWmJ8saQ/vrGm2TFBnoW1WRJktKZ92GpTTC/7ZSh2mC1jvbZgg5gvTFhHO7nKHAI1KDh/6mmF+RdHoKqjhZlxduZ5054/58PcbnuxrOf18/z0QsXunNlV5NLxAhvvO3TyajzktbornD38HidIuDHzO/lHWOUr2ENSPGhKvpTuaq3YY5iFGUK92Yi/mFd6e0WAQRZ9b8xjOhyGoDpnqFMB/iK6XuiVHKsBZrNBRK9DDhpw/N7exS/Jc/YNZedh34fD/nsTGB4PxDQdSprBp9esLw3eMr4bG6n0jAVRM7ujIQ/r8y/m8JPQyqSnDbllcF7FawpC6UEIkQTXUF72U7PAzmJSuCppXZ4v3rUau+auy33XUOc1vKNBU1LssgVRi1FwH6Idmx/ciR9InfgRzBH7+A9s4nMxCcfsIUHo+/QKtX5JoLnO9FhTmaglihR0UiV5Po+XD+zH0Exgtab8WYg5rNyZbZwDUIdShR5iOl+oMuCAKW+kEh7Cf4OpniT6JtHcKA7X9Cuz3UpYISjA3AREAKpjRH3MwoMvSFX7DMwy51fitBKf5yv7+8Stmj6fU6Melx+C0GN3Pq/SZ/ZDQIJcT4Bsxv8ynrcTIau82HW9M0PemryFT0BU2MjEvI4loQNmU99DawSblerlIB5Kj4nyjDJg+N2OeNMErZLFaBUw3gakkm2Eo1lWGOIgJZ1oWMfGc6dldfFplf9OSVnv/CEJiOr6n7qAIoKt84J6MSWxNpGaZIYax0UouSq21Tdi2mgNuSzP2KnucEPKTg0mo7CJCuYbOove0pRnESil7Z6XHI4IESyX78js7JeuB/3v+oLiHlhbcEMeQ29Oz6M/0tSpysSmKhFjgBr3cOrUXtguuj0LIvDs7fVcO5efBhwTvMlhAnqkHLkInoHXGeQkr55klHCl1Hn/++iaUsBj2+/h8CJb0SI/qw4/k4m7e5U/pFck/2iL+HRwO6yPFXSdoeBqgJ4D9gBxDpMHN7e2uuN4fDK4tKj7sHKRLPsvZof84r5Knjs+Bk3GwHKiMStz17WgRvKmZ4foegnVlfEdFslz64zOlHV1B36AE1a4rKgmz5q4aWhf9+F1+PFcVsMWvMwazKgrHW9Oxw0RUvVrFiHDWnGIpqa9PjIXBDBMYwPBej0mQRaeMBe+OGKclRDU3RUN7q1mc+idWfM80wAFkp4HKo96YVO9t6iqvMVt3mWYucTV5E1Z3N519om2546vPOTYAjeUZM06xbstoLZ/QYm1jUKPCl/RcBouxQsEGJa8lScsICAmiPBPnrtuClcmdMzhK4UcAusdUrKPChw6K6P9a77yhU5lqmPe+0q51IGNJocuhRnxO30+8uNlEskrmOpbEEl8Bpi8lRmXpI7IsTIAvWfZXFejX/GWduVKjg2e2CvfsJNauQbC9S9vfvrxh7fvP5xo1qCRD/BHIm/31T905kl5vc6JJuj0zNr2K3XsM02WOcQiEMt9g4OS/anqbO0pzWKsFtO5KsrYXl0FQRVOtanoqV3dCk/V+bdHuDXsmxS8/bIVcEvcc7jRS8Cxau/LKBMrKpAjXzwSd/SsrzQC1p0corIOn63kdUkA65ywLap135Z7QaehpXXqPBiCevHniX88KOQ2OnuTho5Jh9lgwC0K4HvITgxzgShmWZo+CH7QikXfBM53axD9U9d/kpX3QBW8Ogg0PIZ/dIYqVf7M4Ol2nm8V44H2y5YQft1ZyUWQw/LUrBYoq0a+reo4YvdpSBjdzWWfCtxwuxbZ9MbiVG27egfWJWvVOXDaFjE8GXQPFu4/R5VdgP7MNkgLXrOySKCGjfB3FMaCPsI1TmkXMaSh4TADIoLuMWjy8ULHxf2KiZJ2QYMl+sCee+n5Pn/ke2DQdeS7YHKItfiRQxznlpmgNNkXzmiv8BBRD94LE+ziQJug0mMyyEU1tEQsN8qENkSInVSnVz1rOSLbrEzX8P1ONpSxK7NS0j+47M0a16MbvsIoOkbM6VAON4kHBvWG3DJRttAh77F3fYWRrp2oPgdY1nLR/GFRYKrgzHsO02KYNfq+Vw9Yv0aY99X4hy/RGj6zr4MuHL8tnUMXTh1x0tkLI56H7FDgqHPwAxGdpq7ru9Fp5AF3bqRdR/vGvPEBUVP6vnefFZ79MKe8/QuthBGjmWZJa4RDJpjY+jHblg6JBECo2OcQdQxcMQfyg9rn9Z9v/vLOEro8tSMYPXH1vgN9TmX+LVumlXcZHc5IPJnERCfM3puKeLtGq1lDXtIGE/Tt6N36lA9FD0+B0hG0idfScmZtD11BCI32q87kNZVrH1kKq+f3y0yQ8GlaSJnIpCIDC6g6D6uv7zMSrKK0xbh/atYhJEF6hoZch8jX436rbH6KgauLGpeMp5zVChQdIDUTwr74j4lcNwVMAVRw7Y063hQOqkYmuhq4s4O6m9G/h2M3ZJGZxbZnAamghE7+sVjlMzOF/AKLwPHEveEzlzsBJx5DL9q0LAK/QyAKMLcp69shs6obQx1Ug/bNXGav6O0EgwI8B9/xhlDlsDqH9Otb1+wKyAElgEiFDC1fM2VLg+y94gJPJ9GiMJc49wPYyfe+466WLTjveIDPymx1L/tWcXOqzvxq51kFK+5B4mggYFD9zs8XBxLR1tZKHpq7FqGGa5J2tHdTLfTlRPFdHodTclQ2Njt66Xk+vSQpnF8n0RVvbNgWUnd+OZUrAOh19sl+jeS7vDkP5+oEG/uyMKeAaMcFvIPwReC8s93Au65MEZC3ImLMNR1pQHiC6Tq8sG2m19r6ik6qyZPqzfcD2zNYO8TeiTGacTfHBEJ2OMOkr/RNRS/TpbFW8u7rMjTGZjvArXhJ3VJtJSL2VasaRrJbxcp34ZXO5qDo/10CyHGieFnUvpr02DRIKPOCy1M9P1/uwrG60yIczJQcWp7XuRGa1KOWFA4lIg6mwA9lSIZWgu/q+r63HIzB9qOVqXKVnV6ONVcUPpfiCXuRwcvnjrqlDg8UqXennKcYujbunx7nu4QMMFWBb7bqVJfvA4L8kUiB/yo4LvfVWo7W6DvLR0PBo6gR5xptO6pKrEA1MCuJ9KZD4qG5CTigxkcBl3AN+m+blaMDWQbtjVgNaQtw+1CAzopX+3U2jceDjbo71BbqRkP/3PW+LCPrAgR9S1/grMl3f/3+zRma71MGGalOxv+/o83nWZGDPSyo2yKQd6FND1UHwg4OhuIM4vd4YS0JYg3lGUa8lQevOj3DO5acQrn+CEpwp8T8wrmCj+GrM75ypT/e1m+tKwLoxASrScRRnr6jX93uJjvv9T3dX3u/ubFiwZvT5VHYBI0jHiVVl9lPGa48OkbZEk9VS72w6j31koLqAvU1EGWsrwRh9LOHMRa1j1uY2vsRb4aAt4k65NTzdEYjTVJEfq4xx1XNkUVlKTVRP68oo4kGTEAlQVSlrste4h0ddG7IEx/E1OsG8XUizF4/PVGxoJhqA3Cs0fRwrYFjymznqFi5u67Mp8c4Z2N9mJsdMMzFQr8qvfpkBZTExNKHhZdStqExi8KzUFLiJyV6nyH2LyZCC6mHhae9bRwCOSfLVXAxC+WnXBVj37YuR9/34JhYSRzvokVLz2yKpqWzpIAbbd75rdiFMQ4FTMkQysGoLnQHNPTe3yjE+6gQo35ZYh2bwhlWijVveC+Pf7tH/UBHo1if4oj7s6b2hxxUvAsNzpyckKvOrAnQnwBQ16zHx15UKh4g+E4pe/RWC6XGnWDfXNsmaHjVnwwBJrhHZWTdsAgvp3w3jJtYMAFBjwFO9OFDUrfBjdL0oYTpc19JkG/FeCk70M7gBaBfmXXpvlvh3lz5os6kqh9G6qM6EyhDlVvDZG2RHl9zK6wiHV9y5/qgrLbIE5lGoSROLN+q4U2gyhGb+p+wUVLk70DWp9VFw3y2l6+te2I3MVV4OjPFp94l/s/sV7DjsYAm85atnwOmRToIylqOfg6QNVzl1Fsw9avF01Gq1qcui+s0mn8Gfp8OL+DVDwoE7riCQrpI1JaoJ3tLOYpRn4lW9Q65JP5OO3nk255K1BPyhPEmtT/gqaBstdo32eoxAnXS1IAYxuqSQFV3ck9iNiXscCmfiZ/GE5P/7Zl8ruJ6JrSOzO/b/QYQnZT1A55vpGT6BBQ/XQnBXSmvIlN+RfS3H38hn0bl1BRKAjKc50D3ycXH9hh4SdA+ES52/xuy6XrfHso4di/uQysx4ZnnXpgNdJ5jagwPxjjq41iTvxW7H/B2VaEn7lyUz025WyN3eP1ZvUc7Ql8VQq+0HUlVuo90ZD54NfaOas99vYa+uoLYL+yZSE6zqAaJXOuDG9DUncrqSC7pZieu1avsSE6VIAAXvW+IrTkiuC3pVYGMpM9KHOcj3+SsKS0HgDty4uxPLqkj6BVK8+wy6X0wCSOV/6pi8+GkZ6fO/SDYSMUgFg4JSWlomDI0TthPtvfoyfOD+hgU9ZfW91ZGSPOL+rja5EOO+7XA8/hemO5xxJ+YmMUTy7wh0U7xEytFM3N6JhbrFBDveCstZikR15VwA3EsFpNE1GkUZGeGBNP3KeZMFzRHIz3TFX/8JYXQ7N2bD2+/NzVKvHB7djEsBhQMCNPT1+RwN6H9pTdrKvpbnh3fCudpehI+Ka0/CqckLOF2LkiPG9SYB7hB80E7IWWoqovkuzsoBaZsl5Qqtidnr65YxbbrpvSs67GdBC0VLYcgquocMe+SA7Oi5Bf1sjIXqfpYH3paJEt6wyn4+OoaZPqOIV7loL5pOHnT3FLu4hcqGa1z/jgB8kWarutVmkqmdb+kr4hhrUm2Bn9mv+SnFht1M8xogB+yjnWKfG0JjXuBIM9STN97oMyC/SEY+vQOHWygfrgTundg7NSh0kzQH8Wnp7S6nNB1EzOUFly322QgIfQ0SukyERiPsUkif3g77zOw9Se8DNBeFsPeY68Xo9UAvWuo8eZPojITJPjJPW1148BtFUEsMcXDYB0yBObjIBienlOlawGRjPZOzmL+5GEHYETVQ7PW8AX9QVCtUdz4NBHeIIrI5JshDeV1qCUbfvrpZni8OxrD0vHlEYB3GBOlvX8ez8JM5R+CYPw9+Xivo79NPwTLrLSknGG3pTpkrYUMTp7OJ59OL/GTSSjRs7Kb9OAZ2yD9dw146rZtZywOGXf88GGBF7Hw5S/EDWmK+ilNhR9YWZ38D8M3TDQ="
 },
 "experiments/partial-front-sampling-2026-11-24-r29.json": {
  "sha256": "2ab005b9d64bdd388a71a1026df4885152762be1f992cd90607d5256a046468f",
  "size_bytes": 13543,
  "zlib_base64": "eNrtW1uP27oRfj+/gvDz2hZFXRu0QHAStEHbIMgp2ofiQKAoyuaJLKqi5M0myH/vDKmbL5uT3aR9chB4bV5nPs58M7TGn38iZGXEXh54dpStUbpe/YHQO2z+oOoCPqxyrg9crRvedopX67LVdbc2/NBUqt6tm1Z3WuhqZefIj41s1UHWXabs5EdmtX7qJohW8k4WWQGvON73/GhN6doPXH8Hi33Ste17aRTf/kXXu+yv8OL69w+N7vbSKGNHkFz3dSELompc2Uh4Q0AobfpWkk7De2U6kIBwIWQDO5NTCVsl9qRruapxkDrwnTTkwB/gLWh6lKfDSSE7KTpAjfC6IEdlVF7Jda4/kkoLXqlP3HV2hJNSfUTBcEk7WjegnPok23XfoPok74ud7DZOMaP7VsjswGtVStNlZs/9MEIlY5/LhEVhEfK89PIgzOOwCDyP+zIqfJ/SVHiS0TwMpGTMY2Ep80QWgpZpFKd+NAAPIijcdtoC14YGvp26zHYARR6VvF9PZ8Pg/JLtOG/zmwGreWTRhdy5zyTzY8YF9xMZ+iAhyCTLxPPD3EuSRMYs4pyVQez7KYu8ksVx4UXcpzzP09xtMZ5b5oSChT9DM0FL6/aTCq5zkN/8vuQI+CRoUoZlkia+lGmRijzJqSg5jz3BeRCmiaB5EoDQiR8yacUDpIskFx6PeUI9fwULfrHCghWhsWTgYOJDo1W9xEN6nOc0ByWDlMaex2givZwGoghEGCZSMEGjJPCSmFOWx0UhAZZE5kVe+kkQS4dHK4Vq5OBtTm1ZrK35oxuOlmrWRsoiiNZHejJvFqfE7UF+xqkXxhIE4/A/CCRnQqa5z5mMo5gJ4YPVBYJ7ac7SIC+LIhEhQOKWrXguq4y3h6zUbZaDb2YzJwASra6kddXJVAY/H1wuE+DAnZmP1XZMHxFT65PIUiy+Gxt1/hv4IbayOAmm5sEhM1kXFn0cEIHvTAMOyhjcdTkgZv7ULw9N95AttrQdXwaTOfLqqmS+d0WwKPq6XNT346/KRT36mFyDWJPdjTw7w4inD5+CQfWV0Hvddm4FMB5c5N/j4rW8zzz4R/3V3UVbctnms8s2Ri/bguBKW3jZll6OozS6bPOTlW369VQnIyvHyVnbVzZwzJzhJwNvkJdvX5E//ZES4F+9FrqtZUvcYQ1die3aQuw57Ta2fwgDpGy5439Ya8smLkH8pcnA8DPZaLFfmisESTDqrGmlQZ/A2W7k8kzGKIzEhgPbI1jL6aF3bS/Ph+xlVWTgWeB0ouohEF4drq15LQD6Zd+XZSWJrNQODZPUoDbuhm+cWHNQfPPKEGiV5F51e4hrTauOGL7QxNZBRN6//fOGvNcdtsEEScbtDAEticWDdPtW9zv8q8wQGcWDqJQgFQToF6SVTcVhj26vIYhD/FYWJwznEO6JbgtV8/aBOKRbbIBXKxH2230HwUHezerURM5IZum76WRmePyZO/rMcSh6Gptdu5YX3Uk4dXf3l7NpHJ0QCCZPYFLLYwrC80MqdA1UOg1IBglWBSCTGYC5P8t90BoaoPdSwYc5QbqDw+wIZiV5izmXS0DAhu+l2u0xKdqQf+2Bn9cuR5nSJl4ZjQcCmZqZkXdK2Uwm5+LDrsXdN3Pskx+l6DuXUA4kZI/eojgQJEShIc/pWoyTOe8gVFoYJl9pHEBzF4Qn77x3ho/Si84p0QK0ZGMpPpkdVYIX1JhmAjP8p1et5cmFs/C+g/y3UwI8Vh/0oFEJoIwjhig/JHt4FroFP6pBIasxjVkGuTQACC91Vz3Y2Ci52NtwONnmIkkYKcwG5nml1XXYJgGAcsq+qlAjcyYkxHt9OKAERbZrelz47T/fvHrzkvwtgEMlL6nnjctPwdidoUXErjUfLoS+np+eLvDAAQwDgv5HMPy6VKCsQHrxNnS02TGenQ8YjV4dduaTDYbJeIz1wWRK93bYyKAHNAYUcOwY57e8UCprPi7C2eSR1DuJFqO4rSxlOwgyJo9Di9kaMAl8BykwHAPw29Q35ZT0kXzyYv1FruUVMgpyyeI0LoT0ZcAg58vTpPDSyPckz9MohZQ3jP1QFqmMwyjxC1GKImQihjT+fI+v8NkjuUj0vblI+Du5yEhSiu9qbdB9vgr0eHVyFw4wVWB1NK8Z5uA6zNc2WF46ckELwWNaxpDAeqX0yzDgAUspIC3DyBeMhYxHkMszCTcUlkdpnNAU7ksCjohe2ebpYLPvAztNn4/1I6TW10gh1dHeo6ah+n6ZB47qjQoOd4wxS/PiKSNDx9N4FYsXLXNkwqTkZCxczB21vds/GCVsGoW3aSSETnUPd5BL8APR9xA8zV41d0hQkGf1MLLjLVyS1xCYSswPbMzDcATXHSAtQ2bNCC87yAlK+LwnSItgVnUHyZe7fQ8UvW4g+aiIywo35PX4/YC9yEBzx9HvX0DsJHsFd7CaDNmgqgE3AHazVA3SQvzuY0BjaP9y93xEaXCDdAEpDZ6KaXKJ6Xz/u2Fq4XgqpuklpskN0gWkyRMRpd4VK/VukC6t1HsqpvQKm4Y3TJdsGj4R0/BahLpBuoT0qYimN0S/EdGT29si173lsj/wFG4p6/8LuVti+lzkbunnjwMuvJncM/Nzym7IPRO5W4C43Qn/l8jdbn7PRO52v3sucLdr3POAS6IrOcnt6vUtyCU35M6QO/mGYHqIfeQVFpphKUVfiz2vd+dPw+bnkFkrG91i0YCRDccyierhdKzoW/flhJX35KHm9DzwO0sRf3gx4vyI8DHpM2WwTq/SoLJuz1R+956F10sYA98TnlfGTMqc+4EMk5gFfgDGGSd5XKQRi2nBZByEcexTyhIvRdlDKRIascXTVSnUUHu8+lkf8PsfW8bT3evBfN77ydpZxFjbRmwpoTWl0UgISorFtjh3qvQd6m7rQVUw7Q7bzYa8G8qm0Po/yTqKIjI8UbfL7mEBsFJwjyhiZMKKOBsxd3O50VRMDGKSKKJDdfBQaobAKqG66oGMVmXXXwAOdu8qKrDaprqoPBtrbQy5l9ANf3HXBtfXvYF1h6/DQALKCD5DlgZrqGB6QXbolnek0th6h5iOy8IA2O2OhFvqDW7YcNUS04M+BoutsIgMMZwKw9zad+DltgYDq4j6rukRjaJHNbFs2GmHj6rXRkisAHNjNuQf9mDGelACY0Gt8URdgRLwm/N9GP5K26aur9Ee0BB0VcBWrpAISQmLzsxANPAeZUVoYGhfdRvyVpOpfodM9Ts4EbZZgyZodTC6UjxXFTAfERVXh2UZ03EqU/l8RiyqPoISzl0ufN9JYAb33z5SB49sQKkfbIEL1uPCU6n9tMGj7FD4BXh/FHNwLvB4kbDCL4OQs0iIJCoDUXpJWIYQLoooDWhK88DjeSjzJBa+F4Vn7GCr436ANoB0g9a4dis+Kn5MaRxQcKw8T3mS517IuCdpFIR+wLFaPMmTUIg84sAqIhTU5yFolFPqJUEMh3JWTad0mzlRl0LPxUogGPVCJN5RVrZ9//rlq7+/3hyK1ckyg0e53xV820pOa4Blmrud3jkI5gJdoRs5fT+9QjOta+2KQYmLUeDAJ7RienCbo6XJgT0GfjMAfrfFGL+1VZ92RKPBF4HGgEh7Y+mPYDG2I82pHPJnOD5osetNBdl20rJo+4xyX7iVYFnrapBm7AYR5yUcVfP2MG1lPR+SC1ibhvFQYih0NRS6EWWsq3N4N5fKWYJwrOF7pJBHWenGMsfwywyXdJCx2hWJ8jBVqk57/yIXu+Qc8dA1sCY0TMQ2lbfOB2Gu5Skc3updjz8swSJLHPeCqA7hQA0uhQQlXc0kEiMQz0EXGMWAghcH8RJYnbL44pcng34QcVy1MEp+BKpD4DGjIm0P23M72zEusYzrRMd25pPZNAAi9NnDmmMloJWx2etOG/JBygbwsEFlKkue63VPjnGoq3XRAbWWvM0fiE1zNuTNoHSPP4/R0h2rAgN6ODlZwZsOi0sHDjaLHeCQ4P97P12GipofEBc8txeoPky1JmPjBY5dOJCpeQOhorM0/utPX376L5ypp10="
 },
 "results/review/partial-front-sampling-20261124/r28-training-partial-inventory.json": {
  "sha256": "d2d48067a1130a7c83d2f45a36cc86f4cf085f5000d694191b40ab5eb87c2065",
  "size_bytes": 243612,
  "zlib_base64": "..."
 },
 "results/review/partial-front-sampling-20261124/proposed-cohort.json": {
  "sha256": "711741efebb9a8bb053a0e164524a3e328b85ccb6a424c5c12a5f08b110847ce",
  "size_bytes": 32060,
  "zlib_base64": "eNrVXdtuHDmSfe+vEPxslxkkg0EOsAssMM/zA4OFweuOFx6rIWumGzPof19mVbJUIplZcrKU0gqNtlSZWcXg9ZyIE1H//uXu7sPDP77FD3+6+2C9j78+xvDlgesvD/GfX+Nvd//1lz/f/ed/wN2/4sP9J3//8D0+3N27/43+cb6kj5c+33+Pzy//OF7/1T48frXf7tKD9Y9f779P7/VZfPg4fezXv9v/iT/yB/81/3V39+/j/8vrX76GqUnf429fWP4BfnzmeP3H/T8efL7heNuPv1mOarrVKww8BIuRB5cMWJO8jeCN9sJ7iZCMAOMiSeYjs8b4lAi0ZwHQOOYuPiB+i8fWLnTK050xpenOf8Yv36yL3/L9xYp88Vf7+LfpHYJ9tJ9PT/74nB7uvz/++MQZV58APnHxKb/v5+PTPz4/Ptiv3z8/2Xx4/P3x/GFTw87GMqcElxhk4midQ2ld4jyiUWBZtk4oyYLkgrRRaHgUwIWOuQOSogQ6fpjf9I+zKT636+H+21VDvP0evubf4mzLp5Nlzy1S78GiU8PyyP2/H5t5ReU3l+xiwKbV9uO5XSz/qS5aBflvEBcv8PwCh+YT5nX65eH+t6cVebkqTz16/9v0fvzj5YtP7YBnr7v737/8+vuzN5t+pDQH9vHZS8LQgR1/9PMLKnczA2OMQHP8+Vi9lTw9x8XpMl1c/u/z7398XLVG3NgaKfB5q55fzpY0TxAeTrfyLQbIIQOQzZ89/8iqbUqVtp2GCGtzsDYHGY0PC1+aZOwlVikwB6NzV6tqRklg81TrDg5p8bw3PrajfbRMbzFJDJmkWDNvpJ5bC6d/RGUNwqo1mp8uk5q7ZItRODL7yIhTh+Yt8bitVi3ksDr5DPIDIOQJx6/N2i2mqRHTNIhmvFCeWlXPSiPzvSIvRV1P17xZzE9sMYCGDBCwOjaGny7D3MXVxmGEPChi1GzqyMsi2mKSGVlDnM0fXX7qBcP4s+uVyULCQUz7hpgXDFWP87xENxglYGScBNfPjVLVISp02bS645iXSjVPFc0nmFBbrBmCByJPm7o56rxP1evpiuUZVyAcccUWO8SYHVfaptW8fJahzY1mlxxZMkKysgvUa2U+fHR/eygoqHluqxk4BnPooMwFkqQas0CxpsY3zUCg3GjB0HkiFRwIp0UJXVgpsH+24LRPVy8RlHtPQ7fFGBo15oRYqmWByqwtCwS98JyZl1vpnS0m6TGT5l2qz1Uy+1s1jC10iOIzQoJNe/HQcalIrx2HLX6sVrpsECvCjMvmJ7bMPMlGbJJKN+dLQZkNijSNAUrONpuF4fjl8pXZkHXPlr7q2YosEDNWRIrSkAvWM7LaJ8NcJAvSZ3wLQE5OQJgBc5ZTsBhC0kjK0/vzbOll74kErpwlSZJzA0jWaxVdXl3kMRHX3DrHmfMA1jgVTFBOcgKlTEio0L6RZ2tXi17Vs7WrJU+eraeNb9mzhbVni1eeLTBjnq0xV0qL2RAKoeC6dQPVrhSUZbuVW6jREIvIG3LdHgli1TVEqHsMdz4L+P5On3yG1iYglC6t+p9EQ3ukpvPNJ+h6SyfPy/wh1LbqzElnot3wbGxHodDcTQSOyzGXDrTtKYcmntBZhXEMb00w5REtd/dKaYYrc4NOltQmtFinILdt9PPm3qfc9+eJVDUeWoowuBfRsJumpvJlL6pBmugwZTAFpG1qvB5zx7QOjMKFm8V75kELW6zS4nx+3NJb9kIXBrSeofPOwj9eC4QQwDaePATwRbuxE5NlM6n9q1e8NBczSWymlXDjABPKYk7N9RVrF7Jm55t3d+Y9wYF+9+KZYIJq3Rn13Ds7b+X+7rzjkVq3R5W1WfuMOvvXeQuQt3ThvdAzIdrmlH1Hd8656l4059VzS7/dizEda90O/d10Wq0rh94tXXYvbHznEC6hXuR146ld6OoKCPppbwMXV70NwjmNmgRaq1ly2jOKYKJTnqEX6EMek5jICkQXMESWjJQqcMUxaoPvT0eTcdgio1XKZgarAnKSdopVmcxcuTDoo4yQjzlMTMSkbKB8BY23JuY3tE6ZaEjxt/E27GvRa3ob9rXkwttAP+9tMLWMRryljEatH62SmSWQIFohAW6LaAxqZ5RZN0Gq80bf8y+rlnPJM14uZHl/QQ2tQ3p5BmQGe5Eo1TnEeBGtvIFT5dyj/RiAkAsITrN2dM7sAdT++hnd0V3RVfR/Y7+EHuLFuT1acNHIXdS5VV2hj1D6AJJjA1S3xmIHeSW2YRXABSw9xWBWGbIqZGgWQd2ScL7QnBmPzhSx9m6d8TM2IiVaf7CYvmmqDYpKOnxHz9BVd3ys9cKZvTX6ppzzZfz/ikuFQB50b8M6Hjc30iqMnYtEi3HTZo4ALrvu3oBq6q4nkZ+keHXTW5dpEcUwvb9CRKu2OXmq9JqOHZESlHPyFTQVL4tst473qYtnqKFaIcWyT+BGZFPAVbLJvYwqE02jhUaFXGMU3sUQHAs8QyVjvcsNkywYRwgsMeMNN8wnBoaQvTuymTtxkdCQYrld1gmVTHCaYtCJJ2+cTSI40lYYqXwKTJCD5AFSIBEBVErSO/BvQzb3teg1yea+llwkbcB1ssmvJm2oIbbJh9gmkllVkoNciM6QXJegK7a+57xirHWyqSOon2Fo38EoqCUE8hyh2a40l2OGtK3SbAFPk2hPaA4Haczx5t0DlYTrGjN+niB13LtFPpOj21wKG8X+onmQrXNFrBJ+04k2m+IlphOpwd3DmFq06QtaL4TRtKFe9Hu+eTseGgoEgsZeILA4IHr6/tahdHPR9cvYF7RQTi9tsVK2/IuV+WPU7iEbkUEnZNLbSm5y73MFnSjslMcEjGdA103KKkHYTdH8QdV1R9fxlETSxP10G8IRQyMxprIW7CAuMnhqBypKvuC8k60OCdVzZxnsr7BuHdeZMjz3/ny8JvbFvFUNjIcZM6CNBxZZeDsEnagt3yaokGys1W1I+CyL6Hd7Kwy8qpteN2BMQiFM0dZ3l60qeoo2v4gtiPJV2ZFuxJSlvMqUNZgUINmkiKMWijySZErxiBR45sMemU1OeUpCpqCYM4GlPIsoAosa3x1Tzr2+zMYoWEnWSG8gRtSetERAcs6qKCDxSQvPLXMMhWZWKRWiCoSZwLGIwbwNU97Xotdkyvta8sSU+QuYMlRMmWoN+FhYlo+pqEVb3KDslf3MM1It/BclsWaTZmVUBl63huMBLzf7F0TJzkEZ2rTdD7Fh01Glw8HIzgZPvQiyuUFNiaEDF4AfQYHkhLybhnwKBgCr7QGuW4PgNJkYbmL3MORUN6brZu5L8oG1qYpT0GlARA1D4B9YC9ie8nbrvmcrfLMk/W0xgcZM6ES11rPc87Tit6LBizkpL2PwTHW6VPXzXAHaAP9Ks38eouF1iKYCEkgZ8+mIguu8X9q86kKeM8qiU8Llq8kxz4Ch8cmHvJQNDyC0huT1+4NouFLlaKpa5DFIa5y1SstAxpmUiLwMNqWEknllPTMOmfBJaqciB6m1E5wl90YQbVeLXhWi7WrJGETTNUTjbwnRVL8wgTDiwHupelR0aguFUAQfqZYxZgvvE1TJMwDDjtecFDtw3tL154qJ/TGbWbADxQGpk7I0qTjWxqSc0pvGZBi+NaBHCDq7AfULMBuDAQkHwFjzWyG60GbBS9VtvSg+rbkYA70B7pTLuLMBDq0QRZXSBmZ7OYlR7MlFf59SRIduFCbjZ7VUqwUGPOjDCHShgowyuAilxW2Lx4ziUBLPlHq6BdSX0TpTr5ECuJX8ycjGT6NUc92RGKXRqIHnfyD6fLjFIKNDkf/Lf9loNcsWgnPOh5CUlMo7ZtDm6yhEen+SG7PirIomAnitkhTgDHeRkcXEogvJkQouBA4xhciN9UkFmSgYSyl3RGQ6RnoblLpmERMuOutFJBczt0jK5mGxgkseGSPMOI4HLhxXBKCkmMqBaEEuZG7ijdCwL0rdd2wu8jv0z9dJrVHq2V+yMb9Dj1WwxEMNeiRv3BFEcrEGg+ADCdxsrIyBXlX9lAqAS6UDZas9UMXtMoBRxypk6HO+J1uso3chHWpqVXaENwXmbcpNHBMN9UT4pUxDUzSVWC8RfLSE2FiRhrzrLeGjjjCIr7EFKiO7rWjJUI6oKPqnHmbrVo451zRZSBugki20rfjHYF03XMx41bqTz11rxUvGw+56p0kqsFa+rXR6k6iOuPpcEeIJvnuejSSxGvfOIPBwmV4HtYJrodbeORF/W0r1mBiB1Ho60FPbWpHjbXPyx/LaCa6f262UlnUK1I0e9YM57sXT0qdG5/Qt1S0+S3zVtYalxui22hVj6RR6fZu9mD11o9X6g9Js3+TEEKjULYCUpXBmLRQ2fBXDTRlJ83O3oa6ZrVylrokiAtloPKDiiM6wZCGQt8b5FKThwYoUMjmYxDH5KU46eTQY9FTAAN4bdc02r6S/58UhbAxoInGLkURmRSQkBCGz7RQ8uDxJKZ88+Rdg0gHnKvogdV5rhr0Jdd3ZolekrjtbchFgwevUVVTUVdUBlrFkkUHx8gmYqwYrfRJFSGI0TCVLTfxU11DHuSKT0XXKGsz6U9wEpYb2zalRFZ1rleJSHXBKdFTUdQKKg1BbT7JRZ3gDysWhFOBleXcErtuByBP2hEcU1UGkGdFLJuV2LTCMYds56Vpp2SCOE+qFtuLm6cQyzYWT/lMbiWp3r8IkV5JiJbPqNOu5Fn3x1Qwalaz5o8QTPuFyW7GFoSRsfIpEdH0lJWW+3iGOCVY9NizLF7NsK7swVpRTHXsSVFvzTiyExQwcdzHORVOofA7FaLUpN37M6XMstDlxI71Q0HvufU3dUqPATl8MAQXO836yQ94B+S39QezFur/+1GGHBUfJwlcUIMCLnI0/DXH59VrfAVMGuNwmHgJZ7gMzwosJvLKomcDohMlIIskgVZAZboBwghsUEKNCenfRmSkYvgijQmIs5RUhlAZngQntSBhvY8ym24SGhfz/oC1PGWQlAATpTQJrooncv031rZ0tek2Iu68lF9GZF0Bcea361piGaDCm3SbnFIEH7wkkun5qOVDrwty4mKVk+kDQOcmmIlvUqrs5XECW/SXSnS8tOuZL9SrTKoCFg8HIA+LuAptj2bNWrqTVxbfSiU6p0dqfqQ5yc2FgGHNe8iW5lpB9Xz8VN+DCFzkqKjqIgZDlGMjLiJVYL/g1ZR1SL0rZ+UK+PLR6Whk0EqWUYwnoC9vNudoUnxtXxyelPmhcJiKGDkQX4U2xO8LTnC8t5BKMpC7QM8Ic+DI1x+l9d49kXOnvS55Q69WesrsbdjFnDomNHvNB5aA4zj0JvElmLfXd4JR43zobOjJCOnFIkG9QuRrgVDh4ojzQi2vMVR4zLH+BnhPnpAWlYSD3WI4FnNbrOKqFHFiS3e93faPaAuqJ5PXjRbyPwpapXhFvr5ftes70fpl+++OX/wMAupaq"
 }
}
BASE_HELPER_SHA256 = {
 "scripts/pose_checkpoint.py": "2d1daf5f3a228d712d0ac40fbcc2ccb390c1f20e15b61c5466eb732de33c4bc8",
 "scripts/stage_pose_training.py": "7f66075adb0e6e7f279d2e658cb39ef16df30fad050eab07abe25342c316c7a5",
 "scripts/verify_pose_candidate.py": "03395b9a82a4c120647a1cac51c89fd214d8cf9b15c04e2196a40f39077d8491",
 "scripts/verify_pose_box_candidate.py": "f8613b9b310f25930a9d47169650473d3fbc0a1032868dd80222d956d26e1ea5",
 "scripts/verify_pose_annotation_candidate.py": "0dc3d263b4aad594e1019de8f8318dcd963d9c0c2f4604ad2dee408c83af7979",
 "scripts/verify_pose_endpoint_candidate.py": "9adcf9805ba9ee6fbe252c05903bab5a997e9e7c5063362d77e1f5fcccdb53d5",
 "scripts/verify_pose_composed_candidate.py": "e538dfce5de0b1fb10047bbce0598882ee1083825ed6061469c85a3daf08bd32",
 "scripts/verify_observed_training_candidate.py": "28fea1d483058988ea567d9382ed82f73e511cd9b692af7bb8c23c3463134426",
 "scripts/compose_observed_pose_review.py": "8f99c87236b09ed40c85521b68ea2a39ca4062cfa74b8cd43bb376cb5009fd25",
 "scripts/verify_incremental_pose_candidate.py": "ce971a4594f10292ceb61f206f83694fa2ae8bad477442fcc6fc261d3e07aea7",
 "scripts/compose_incremental_pose_candidate.py": "9b51b8a96407af15fa820b959d30ccffe3da21ca53b4ba1f91294bb46360a789",
 "scripts/verify_front_review_r4.py": "b46393cb5a186fd687ed6e2a4e1141711e319e138f361bd6fcbe254effb0afa0",
 "scripts/verify_front_review_r5.py": "74b0108ed6bcd109dae0cb278b23fd6cc8f44cbbfb20c65d4ce1fa9e53570d94",
 "scripts/verify_front_review_r6.py": "2591cc62d1c9238971358d8d1a9f20f324684a4f125aa63b29a1326c4bd00c82",
 "scripts/verify_front_review_r7.py": "afc116835e371dd459bcf62274ed90124196cc7ccb15f580524322b0d7c85fee",
 "scripts/verify_front_review_r8.py": "49675f497acb6fab08d473d506f30dea7b1fa20633bfc10a77c0c6ba03e64c30",
 "scripts/front_review_experiment.py": "58fc3f020edfe834c4d051ba98a954661ffaf7bdf2ce2dbc0caf2fbec3a7f524",
 "scripts/front_review_r27.py": "9525313fd817ac7177daf73e07da951924ab862112bbf0a5870225aa649d7a07",
 "scripts/front_review_r28.py": "8cee625eff3c8c49064901e26fbf56f90ba8e65808ecf99d935ef95bf874c0a5",
 "scripts/front_review_r28_training.py": "8426f8fbcd3e872eb5ed58e88dc86cfb9788223830b3ffb0a42c9d768a016fa0",
 "scripts/evaluate_pose.py": "3f80bd09efbe97327b705ffe6ad3e662111f3e2bc71b69b51a9d38d7baa90ce0",
 "scripts/pose_scoring.py": "04e5cdbef26df978707d0b6e4b82f05471b6012845f7986b9c05c71644ec0018"
}
ASSEMBLY_NAME = 'R29 notebook assembly'
ASSEMBLY_EMAIL = 'r29-notebook@localhost'
ASSEMBLY_DATE = '2026-11-24T00:00:00+00:00'
ASSEMBLY_MESSAGE = 'Assemble verified R29 notebook sources locally'


## 1. Prepare the verified source and original inputs

This standalone file fetches the already published R28 project base. It includes
the exact new training code and sampling specification, verifies their hashes,
and records them in a deterministic local commit. This keeps the clean-source
checks and makes saved runs reproducible without a new GitHub publication.
It does not push or upload anything to GitHub.

Then it obtains the original photos and initializer using the existing published
repository. The accepted R28 labels and staging policy remain the training inputs.


In [ ]:

lfs = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
if lfs.returncode:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "git-lfs"])
git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0"}
assert not REPO.is_symlink(), f"Source folder is a symlink: {REPO}. Choose a fresh RUN_BASE before continuing."
if not REPO.exists():
    run(["git", "init", REPO], env=git_env)
    run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], cwd=REPO)
    run(["git", "fetch", "--depth=1", "origin", BASE_REVISION], cwd=REPO, env=git_env)
    run(["git", "checkout", "--detach", BASE_REVISION], cwd=REPO, env=git_env)
head_probe = subprocess.run(["git", "rev-parse", "--verify", "HEAD"], cwd=REPO,
                            capture_output=True, text=True)
assert head_probe.returncode == 0, (f"Source checkout is incomplete at {REPO}. "
    "Preserve or rename only that source folder, then rerun. Keep the run and Drive folders intact.")
actual_revision = head_probe.stdout.strip()
assert actual_revision in (BASE_REVISION, SOURCE_REVISION), "Existing source belongs to different pins; preserve it and choose a new RUN_BASE."
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip(), "Existing source has local changes; preserve it before continuing."
for relative, expected in BASE_HELPER_SHA256.items():
    assert sha256(REPO / relative) == expected, f"Published helper differs: {relative}"
if actual_revision == BASE_REVISION:
    for relative, spec in EMBEDDED_FILES.items():
        path = REPO / safe_member(relative)
        assert not path.is_symlink() and path.resolve().is_relative_to(REPO.resolve())
        payload = zlib.decompress(base64.b64decode(spec["zlib_base64"], validate=True))
        assert len(payload) == spec["size_bytes"] and hashlib.sha256(payload).hexdigest() == spec["sha256"]
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(payload)
    run(["git", "-c", "core.autocrlf=false", "add", "--", *EMBEDDED_FILES], cwd=REPO)
    source_commit_env = {**os.environ,
        "GIT_AUTHOR_NAME": ASSEMBLY_NAME, "GIT_AUTHOR_EMAIL": ASSEMBLY_EMAIL,
        "GIT_COMMITTER_NAME": ASSEMBLY_NAME, "GIT_COMMITTER_EMAIL": ASSEMBLY_EMAIL,
        "GIT_AUTHOR_DATE": ASSEMBLY_DATE, "GIT_COMMITTER_DATE": ASSEMBLY_DATE}
    run(["git", "-c", "commit.gpgsign=false", "commit", "-m", ASSEMBLY_MESSAGE],
        cwd=REPO, env=source_commit_env)
actual_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual_revision == SOURCE_REVISION, "Local source assembly differs from the verified commit."
assert subprocess.check_output(["git", "rev-parse", "HEAD^"], cwd=REPO, text=True).strip() == BASE_REVISION
for relative, spec in EMBEDDED_FILES.items():
    assert sha256(REPO / relative) == spec["sha256"], f"Embedded source differs: {relative}"
assert sha256(REPO / PROTOCOL_PATH) == PROTOCOL_SHA256
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Verified local training source. No new GitHub publication is needed.")

# Original R28 data and initializer
run(["git", "lfs", "install", "--local"], cwd=REPO)
reference_manifest = json.loads((REPO / "data/manifest.json").read_text())
expected_images = {"data/" + row["image"] for row in reference_manifest["records"]}
tracked_images = set(subprocess.check_output(
    ["git", "ls-files", "data/images/train/*.png", "data/images/val/*.png"], cwd=REPO, text=True).splitlines())
assert len(expected_images) == 157 and tracked_images == expected_images
assert reference_manifest["model_checkpoint"] == "weights/cassette_lower_epoch173.pt"
assert reference_manifest["model_epoch"] == 173
run(["git", "lfs", "pull", "--include=data/images/train/*.png,data/images/val/*.png,weights/cassette_lower_epoch173.pt", "--exclude="], cwd=REPO)


## 2. Prepare the GPU and verify the sampling plan

Keep Colab's matching CUDA PyTorch/torchvision installation and install the pinned trainer. The preflight checks the actual R28 training supervision, the selected cohort and the complete 30-epoch exposure plan. It does not rerun the completed PR105 evaluation.

All training settings remain fixed. In each candidate epoch, omitted ordinary images rotate through 83 eligible photos; each still receives 26 or 27 presentations over the full run. Each selected photo receives 60, and each protected photo receives 30.


In [ ]:
import torch
print("Runtime:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (completed-pair recovery only)")
from importlib.metadata import version
constraints = RUN_BASE / "torch-constraints.txt"
constraints.write_text("".join(f"{name}=={version(name)}\n" for name in ("torch", "torchvision")), encoding="utf-8")
run([sys.executable, "-m", "pip", "install", "-c", constraints, "ultralytics==8.4.182", "psutil"])
run([sys.executable, "-c", "import torch, torchvision, ultralytics; "
     "assert ultralytics.__version__ == '8.4.182'; "
     "print('Ready:', torch.__version__, torchvision.__version__, ultralytics.__version__)"])
import psutil
HOST_RSS_LIMIT_MIB = min(12000, int(psutil.virtual_memory().available / 2**20 * 0.80))
print("Host-RAM stop budget (not GPU VRAM):", HOST_RSS_LIMIT_MIB, "MiB")
print("Limits:600 seconds for the trial;21600 seconds for each complete run.")

sys.dont_write_bytecode = True
sys.path.insert(0, str(REPO / "scripts"))
from evaluate_pose import preflight_dataset
from stage_pose_training import source_state
from partial_front_sampling import (load_protocol, sampling_inventory, planned_exposure,
                                    experiment_identity, source_identity, verify_run,
                                    verify_trial, package_pair)
CONTEXT = load_protocol(REPO, PROTOCOL_PATH)
assert CONTEXT["sha256"] == PROTOCOL_SHA256
assert CONTEXT["document"]["experiment_id"] == EXPERIMENT_ID
PREFLIGHT = preflight_dataset(REPO, require_weights=True)
assert PREFLIGHT["model_sha256"] == CONTEXT["document"]["initial_checkpoint_sha256"]
STATE = source_state(REPO, REPO / CONTEXT["document"]["candidate_manifest"])
assert STATE["counts"]["candidate"] == CONTEXT["document"]["training_counts"]
INVENTORY = sampling_inventory(REPO, CONTEXT)
PLAN = planned_exposure(REPO, CONTEXT)
assert INVENTORY["counts"]["images"] == 137 and INVENTORY["counts"]["objects"] == 3784
assert len(INVENTORY["cohort_image_ids"]) == 9
assert len(INVENTORY["protected_image_ids"]) == 45
assert len(INVENTORY["omission_pool_image_ids"]) == 83
for role in ROLES:
    assert PLAN["roles"][role]["images"] == 4110
    assert PLAN["roles"][role]["batches"] == 1050
    assert set(PLAN["roles"][role]["per_image"]) == set(INVENTORY["image_ids"])
PLAN_PATH = RUN_BASE / "sampling-plan.json"
plan_bytes = (json.dumps({"protocol_sha256": PROTOCOL_SHA256, "inventory": INVENTORY,
                         "planned_exposure": PLAN}, indent=2, allow_nan=False) + "\n").encode()
assert not PLAN_PATH.exists() or PLAN_PATH.read_bytes() == plan_bytes
PLAN_PATH.write_bytes(plan_bytes)
save_to_drive(PLAN_PATH)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Verified accepted R28 training:", INVENTORY["counts"])
print("Extra-exposure photos:", ", ".join(INVENTORY["cohort_image_ids"]))
print("Both roles: 30 epochs, 1050 batches, 4110 images; expected optimizer updates: 78.")
for role in ROLES:
    print(role, "planned object presentations (0/1/2 locatable corners):",
          PLAN["roles"][role]["object_presentations_by_locatable_corners"])


## 3. Run the resource trial and matched comparison

The four-batch trial checks the real training path. Its weights are discarded. The control and candidate each start again from original epoch173 and use the same GPU/software runtime. Progress appears while training.

Completed models are verified before reuse and backed up to Drive. If a run was interrupted, its incomplete folder is preserved with the log; the message identifies it. A different runtime cannot silently complete the other half of a saved pair.


In [ ]:
def assert_finished(output, role, *, trial=False):
    bundle = (verify_trial(REPO, output, role, CONTEXT) if trial
              else verify_run(REPO, output, role, CONTEXT))
    receipt = bundle["receipt"] if "receipt" in bundle else bundle
    assert receipt["source"] == source_identity(REPO)
    assert receipt["experiment"] == experiment_identity(CONTEXT, role)
    assert receipt["runtime"]["device"].startswith("cuda:")
    return receipt

def assert_current_runtime(receipt, current, label):
    expected = receipt["runtime"]
    assert all(expected[k] == v for k, v in current.items() if k != "cudnn_version"), f"Current runtime differs from {label}; select a matching runtime before full training."
    assert expected["backend"]["cudnn_version"] == current["cudnn_version"], f"Current cuDNN differs from {label}."

def train_one(role, *, trial=False):
    folder = "resource-trial" if trial else role
    output = RUN_BASE / folder
    if output.exists() or output.is_symlink():
        try:
            receipt = assert_finished(output, role, trial=trial)
        except Exception as error:
            raise FileExistsError(f"Preserve incomplete or incompatible run {output}. Archive only this incomplete folder, then retry with the same workspace. {error}") from error
        print("Reusing verified completed run:", folder, flush=True)
        save_to_drive(output)
        return output, receipt
    command = [sys.executable, "-B", "scripts/train_pose_sampling.py", "--root", str(REPO),
               "--protocol", PROTOCOL_PATH, "--trial" if trial else "--train", "--role", role,
               "--device", "0", "--threads", "4", "--trial-batches", "4",
               "--max-seconds", "600" if trial else "21600",
               "--max-rss-mib", str(HOST_RSS_LIMIT_MIB), "--output", str(output)]
    print("Starting", folder, "with accepted R28 labels. Log:", output / "train.log", flush=True)
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, start_new_session=True)
    descendants = {}
    def remember_descendants():
        try:
            for child in psutil.Process(process.pid).children(recursive=True):
                descendants[child.pid] = child
        except psutil.NoSuchProcess:
            pass
    started, last_notice = time.monotonic(), -30.0
    try:
        while process.poll() is None:
            remember_descendants()
            elapsed = time.monotonic() - started
            if elapsed - last_notice >= 20:
                progress_file = output / "training-progress.json"
                if progress_file.exists():
                    progress = json.loads(progress_file.read_text())
                    print(f"{folder}: {elapsed:.0f}s; {progress.get('status')}; epochs {len(progress.get('completed_epochs', []))}/30; batches {progress.get('batches_completed', 0)}; updates {progress.get('optimizer_steps', 0)}", flush=True)
                else:
                    print(f"{folder}: verifying and staging inputs ({elapsed:.0f}s)", flush=True)
                last_notice = elapsed
            time.sleep(1)
        captured, _ = process.communicate()
        print(captured)
        if process.returncode:
            log = output / "train.log"
            if log.exists():
                print("\n".join(log.read_text(errors="replace").splitlines()[-45:]))
            raise RuntimeError(f"{folder} failed; preserve its log and receipt. No partial run is accepted.")
    except BaseException:
        remember_descendants()
        children = list(descendants.values())
        for child in reversed(children):
            try:
                child.terminate()
            except psutil.NoSuchProcess:
                pass
        if process.poll() is None:
            process.terminate()
        _, alive = psutil.wait_procs(children, timeout=5)
        for child in alive:
            try:
                child.kill()
            except psutil.NoSuchProcess:
                pass
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    receipt = assert_finished(output, role, trial=trial)
    save_to_drive(output)
    return output, receipt

COMPLETED = {}
for role in ROLES:
    restore_from_drive(role)
    output = RUN_BASE / role
    if output.exists() or output.is_symlink():
        _, COMPLETED[role] = train_one(role)

TRIAL_RUNTIME = None
if len(COMPLETED) == 2:
    assert COMPLETED["control"]["runtime"] == COMPLETED["candidate"]["runtime"]
    print("Both roles are complete; no training will repeat.")
else:
    assert HOST_RSS_LIMIT_MIB >= 6000, "Choose a runtime with more free host RAM."
    probe = r"""
import importlib.metadata as m, json, platform, torch
assert torch.cuda.is_available()
assert (torch.ones(1, device='cuda:0') * 2).item() == 2
print(json.dumps({'python':platform.python_version(),'torch':torch.__version__,
'ultralytics':m.version('ultralytics'),'torch_cuda':torch.version.cuda,'platform':platform.platform(),
'device':'cuda:0','device_name':torch.cuda.get_device_name(0),'device_capability':list(torch.cuda.get_device_capability(0)),
'packages':{n:m.version(n) for n in ('torchvision','numpy','Pillow','opencv-python','psutil')},
'cudnn_version':torch.backends.cudnn.version()}))
"""
    current = json.loads(subprocess.check_output([sys.executable, "-c", probe], text=True, cwd=REPO))
    for role, receipt in COMPLETED.items():
        assert_current_runtime(receipt, current, f"completed {role}")
    TRIAL, trial_receipt = train_one("candidate", trial=True)
    assert_current_runtime(trial_receipt, current, "resource trial")
    TRIAL_RUNTIME = trial_receipt["runtime"]
    for role, receipt in COMPLETED.items():
        assert receipt["runtime"] == TRIAL_RUNTIME, f"Trial differs from completed {role}; no full role started."

CONTROL_RUN, control_receipt = train_one("control")
if TRIAL_RUNTIME is not None:
    assert control_receipt["runtime"] == TRIAL_RUNTIME, "Control runtime differs from trial."
CANDIDATE_RUN, candidate_receipt = train_one("candidate")
assert control_receipt["runtime"] == candidate_receipt["runtime"], "The two roles use different runtimes."
assert control_receipt["optimizer_steps"] == candidate_receipt["optimizer_steps"] == 78
assert control_receipt["batches_completed"] == candidate_receipt["batches_completed"] == 1050
assert control_receipt["images_processed"] == candidate_receipt["images_processed"] == 4110
print("Both roles completed with the same training budget. Package the verified artifacts next.")


## 4. Save and return the result

The ZIP contains both final checkpoints, training receipts, source and label identities, the fixed experiment protocol, complete per-image sampling exposure and epoch results. Packaging checks equal update budgets and runtime before calling this a matched pair. Keep the Drive copy as well as the downloaded ZIP.


In [ ]:
RESULTS_ZIP = RUN_BASE / "baomai-partial-front-sampling-r29-results.zip"
restore_from_drive(RESULTS_ZIP.relative_to(RUN_BASE).as_posix())
result = package_pair(REPO, CONTROL_RUN, CANDIDATE_RUN, RESULTS_ZIP,
                      verify_existing=RESULTS_ZIP.exists(), protocol=PROTOCOL_PATH)
print(json.dumps(result, indent=2))
save_to_drive(RESULTS_ZIP)
try:
    from google.colab import files
except ImportError:
    print("Completed results:", RESULTS_ZIP)
else:
    files.download(str(RESULTS_ZIP))


### How the returned models will be judged

Compare both fresh models at the unchanged confidence 0.13, NMS IoU 0.70, matching IoU 0.50 and endpoint-confidence 0.50 settings. Preserve the frozen 666 primary and historical 663 reports, with the accepted R28 661-front review explicitly separate.

Report recall for all zero-, one- and two-corner fronts; gained/lost identities; both-corner success within 5/10 original pixels with eligible misses retained; unmatched outputs, duplicates and empty-scene outputs. The thirteen previously diagnosed misses are a useful cohort, not the only outcome. More partial-front matches must be considered alongside ordinary-front preservation.

No confidence/seed/epoch search follows this single run. These repeatedly reviewed development photos do not establish performance in a new capture session or metric robot-grasp accuracy. Keep PR35's selected status until the established comparison policy supports a change.
